# CNN + BiMamba with class-conditional influence (influence_cc) — ARI run, seed 42, full report version

This is the **same code as the ARI run** (notebook38d5af95dc): same data, same leakage-free split,
same model, same hyperparameters, `SEED = 42`, same XAI evaluation.

**Only additions** (they do not change the model or the results):

1. The training loop also records **train accuracy, validation loss and validation accuracy** every epoch
   → training curves for the overfitting check.
2. Extra report cells that save every figure and table used in the faculty report
   (all file names start with `report_`).

| Report item | File |
|---|---|
| Dataset samples (5 classes) | `report_fig_dataset_samples.png` |
| Augmentation examples | `report_fig_augmentation.png` |
| System diagram (black and white) | `report_fig_system_diagram.png` / `.svg` |
| Training curves (loss + accuracy, train vs validation) | `report_fig_training_curves.png`, `report_training_history.csv` |
| Confusion matrix | `report_fig_confusion_matrix.png` |
| Classification report | `report_table_classification.csv`, `report_table_overall.csv` |
| Precision / recall / F1 chart | `report_fig_precision_recall_f1.png` |
| Misclassification report | `report_table_misclassification.csv`, `report_fig_misclassification.png`, `report_fig_misclassified_examples.png`, `report_misclassified_images.csv` |
| XAI heatmaps (influence_cc, Grad-CAM, LIME, MambaLRP; one image per class) | `report_fig_heatmaps.png` |
| XAI stability plot / faithfulness curves / summary table | `report_fig_stability.png`, `report_fig_faithfulness.png`, `report_table_xai.csv` |

**Kaggle settings:** attach LC25000, Accelerator = GPU T4, Internet = ON. Then *Save Version → Save & Run All* (about 4 h).

## Section 1 — Configuration

One model, one split, six explanation methods. Every knob is here.

**Run once with `QUICK_TEST = True`** (a few minutes) to confirm nothing crashes, then set it
to `False` and use *Save Version → Save & Run All (Commit)* so it survives you closing the laptop.

In [1]:
QUICK_TEST = False       # True = few minutes, tiny subset. Run this FIRST.
                         # False = the real run (~4 h)

# ---------------- data / training ----------------
MODEL_NAME, SPLIT_NAME = "cnn_mamba", "refgroup"
EPOCHS        = 1 if QUICK_TEST else 12
BATCH_SIZE    = 8 if QUICK_TEST else 16      # lower this first if you hit CUDA OOM
LR            = 3e-4
WEIGHT_DECAY  = 0.05
IMG_SIZE      = 224
NUM_CLASSES   = 5
SEED          = 42
MAX_PER_CLASS = 60 if QUICK_TEST else 5000

# ---------------- architecture ----------------
D_MODEL, DEPTH, D_STATE, D_CONV, EXPAND, PATCH = 192, 4, 8, 4, 2, 2
CHUNK_L        = 32      # scan chunk; smaller = less GPU memory
USE_CHECKPOINT = True    # recompute activations in backward; ~12x less training memory

# ---------------- XAI evaluation ----------------
PER_CLASS      = 2 if QUICK_TEST else 40      # XAI images sampled PER CLASS
N_XAI_IMAGES   = PER_CLASS * 5                # 5 classes -> 10 or 200 in total
LIME_SAMPLES   = 40 if QUICK_TEST else 150
DELETION_STEPS = 10 if QUICK_TEST else 25
TOP_FRAC       = 0.20
N_BOOTSTRAP    = 200 if QUICK_TEST else 10000

REFERENCE = "influence_cc"     # every significance test is against this method
ALPHA     = 0.05

OUT_DIR = "/kaggle/working/results"

# ---------------- parent groups ----------------
REF_URL = ("https://raw.githubusercontent.com/GeorgeBatch/LC25000-clean/main/"
           "kaggle/lc25000_image_groups.csv")
SPLIT_ON = "reference"            # "reference" = published LC25000-clean groups (recommended)
                                  # "kmeans"    = our own clustering (leaks; audit only)
USE_SYNTHETIC_IF_MISSING = True   # lets the notebook run even without the dataset attached

print(f"QUICK_TEST={QUICK_TEST}  epochs={EPOCHS} batch={BATCH_SIZE} "
      f"max/class={MAX_PER_CLASS} n_xai={N_XAI_IMAGES}")
if not QUICK_TEST:
    print("\nexpected wall-clock on a Tesla T4:")
    print("   parent grouping     ~9 min")
    print("   training, 12 epochs ~145 min")
    print("   per-image XAI       ~85 min   (7 methods, 2 sanity scopes; LIME dominates)")
    print("   statistics          <1 min")
    print("   total               ~4 h      (Kaggle limit is 12 h)")

QUICK_TEST=False  epochs=12 batch=16 max/class=5000 n_xai=200

expected wall-clock on a Tesla T4:
   parent grouping     ~9 min
   training, 12 epochs ~145 min
   per-image XAI       ~85 min   (7 methods, 2 sanity scopes; LIME dominates)
   statistics          <1 min
   total               ~4 h      (Kaggle limit is 12 h)


## Section 2 — Imports, seeds, GPU

Every reachable seed is fixed. This matters more than usual here: one of the claims concerns
the *variability* of explanations, which cannot be measured on top of uncontrolled noise.

In [2]:
import os, math, random, time, warnings, glob, copy
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
from collections import defaultdict

import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(OUT_DIR, exist_ok=True)
def free_gpu():
    if DEVICE == "cuda": torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
def gpu_peak_gb():
    return torch.cuda.max_memory_allocated()/1024**3 if DEVICE == "cuda" else 0.0

print("torch:", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda": print("gpu  :", torch.cuda.get_device_name(0))
print("output:", OUT_DIR)

torch: 2.10.0+cu128 | device: cuda
gpu  : Tesla T4
output: /kaggle/working/results


## Section 3 — Dataset

LC25000's five classes. We search all of `/kaggle/input` for the folder names so the notebook
survives a change in the dataset's directory layout.

`lung_aca` versus `lung_scc` is the pair that every recent paper in this area still confuses.
It gets its own analysis in Section 7.

In [3]:
CLASSES = ["lung_aca","lung_scc","lung_n","colon_aca","colon_n"]
CLASS_TO_IDX = {c:i for i,c in enumerate(CLASSES)}

def find_class_dirs(root="/kaggle/input"):
    found = {}
    if not os.path.isdir(root): return found
    for dp, dns, _ in os.walk(root):
        for d in dns:
            if d in CLASSES and d not in found: found[d] = os.path.join(dp, d)
    return found

class_dirs = find_class_dirs()
SYNTHETIC = len(class_dirs) < len(CLASSES)
SYN_DIR = "/kaggle/working/_syn"

if SYNTHETIC:
    if not USE_SYNTHETIC_IF_MISSING:
        raise FileNotFoundError("LC25000 not found. Attach it via 'Add Input'.")
    print("!! LC25000 NOT FOUND -> synthetic images.")
    print("!! The pipeline will run, but every number is meaningless.")
    rng = np.random.default_rng(SEED); rows = []
    npar = max(2, max(20, MAX_PER_CLASS)//20)
    for ci, c in enumerate(CLASSES):
        d = os.path.join(SYN_DIR, c); os.makedirs(d, exist_ok=True)
        for p in range(npar):
            base = rng.integers(0,255,(96,96,3),dtype=np.uint8)
            base = (base*0.4 + np.array([200-30*ci,150,190])*0.6).astype(np.uint8)
            for k in range(20):
                img = np.rot90(base, k%4)
                if k >= 10: img = img[:, ::-1]
                fp = os.path.join(d, f"p{p:04d}_c{k:02d}.png")
                if not os.path.exists(fp):
                    Image.fromarray(np.ascontiguousarray(img)).resize((128,128)).save(fp)
                rows.append({"path":fp,"cls":c,"y":ci})
    df = pd.DataFrame(rows)
else:
    print("LC25000 found:")
    rows = []
    for c in CLASSES:
        fs = sorted(glob.glob(os.path.join(class_dirs[c],"*.jpeg")) +
                    glob.glob(os.path.join(class_dirs[c],"*.jpg")))[:MAX_PER_CLASS]
        print(f"  {c:10s} {len(fs):5d}")
        rows += [{"path":f,"cls":c,"y":CLASS_TO_IDX[c]} for f in fs]
    df = pd.DataFrame(rows)

df = df.reset_index(drop=True)
print("total images:", len(df))

LC25000 found:
  lung_aca    5000
  lung_scc    5000
  lung_n      5000
  colon_aca   5000
  colon_n     5000
total images: 25000


## Section 4 — Leakage: recovering parent images, and checking them (ARI audit)

**This is why published LC25000 accuracies are inflated.**

The 25,000 images come from about 250 original tiles per class. Each original was expanded by
rotations (up to ±25°) and horizontal/vertical flips into about 20 near-identical children.
Under a random split almost every test image has a sibling in training. The model can then
*recognise* instead of *generalise*.

### Step 1: our own guess (KMeans)

We describe every image with features that do not change under the augmentations that were applied:

| feature | why it is invariant |
|---|---|
| HSV colour histogram | rotating or flipping moves pixels but does not change their colours |
| radial intensity profile | brightness averaged over rings around the centre is unchanged by rotation or flip about the centre |

Then `KMeans(k=250)` per class. This is only approximate. Two different originals from the same
class, stained in the same batch, can have near-identical colour histograms. Normal tissue is
especially uniform. A ±25° rotation also changes which tissue sits in the corners, so the ring
profile is not perfectly invariant.

### Step 2: the check (ARI against LC25000-clean)

The LC25000-clean project (GitHub `GeorgeBatch/LC25000-clean`) published a parent-group label for
every image. Cite the paper named in that repository's README. We join on file name and measure:

* **ARI** (adjusted Rand index): 1 = identical grouping, 0 = chance.
* **NMI, homogeneity, completeness**: shared information; "is each of our groups one parent?"; "is each parent inside one of our groups?".
* **Pair recall**: of image pairs that truly share a parent, the fraction we grouped together. **This is the one that matters for leakage.** Every true pair we separate can straddle the train/test boundary.

Groups never cross classes, so the global row is inflated by class agreement. **Read the per-class rows.**

### Step 3: the split we train on

We split on the **published** groups, so no original image is shared between train, validation
and test. The KMeans split is still built, but only to report how much it would have leaked.

In [4]:
from sklearn.cluster import KMeans
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score,
                             homogeneity_completeness_v_measure)
from scipy.special import comb
import urllib.request

def invariant_features(path, size=64, n_rings=16, n_bins=16):
    """Rotation/flip-invariant descriptor: HSV histogram + radial intensity profile."""
    im = Image.open(path).convert("RGB").resize((size,size))
    a = np.asarray(im, np.float32)/255.0
    hsv = np.asarray(im.convert("HSV"), np.float32)/255.0
    hist = np.concatenate([np.histogram(hsv[...,c], bins=n_bins, range=(0,1),
                                        density=True)[0] for c in range(3)])
    gray = a.mean(-1)
    yy, xx = np.mgrid[0:size,0:size]
    r = np.sqrt((yy-size/2)**2 + (xx-size/2)**2)
    r = (r/(r.max()+1e-8)*(n_rings-1)).astype(int)
    radial = np.array([gray[r==k].mean() if (r==k).any() else 0.0 for k in range(n_rings)])
    f = np.concatenate([hist, radial]).astype(np.float32)
    return f/(np.linalg.norm(f)+1e-8)

print("step 1: KMeans parent groups (a few minutes on the full dataset) ...")
t0 = time.time(); df["group_kmeans"] = ""
for c in CLASSES:
    idx = df.index[df.cls==c].to_numpy()
    feats = np.stack([invariant_features(p) for p in df.loc[idx,"path"]])
    k = int(min(250, max(2, len(idx)//2)))
    km = KMeans(n_clusters=k, n_init=4, random_state=SEED).fit(feats)
    df.loc[idx,"group_kmeans"] = [f"{c}_{g}" for g in km.labels_]
    print(f"  {c:10s} {len(idx):5d} images -> {k:4d} KMeans groups")
print(f"done in {time.time()-t0:.0f}s   unique KMeans groups: {df.group_kmeans.nunique()}")

step 1: KMeans parent groups (a few minutes on the full dataset) ...
  lung_aca    5000 images ->  250 KMeans groups
  lung_scc    5000 images ->  250 KMeans groups
  lung_n      5000 images ->  250 KMeans groups
  colon_aca   5000 images ->  250 KMeans groups
  colon_n     5000 images ->  250 KMeans groups
done in 1649s   unique KMeans groups: 1250


In [5]:
# ---------- step 2a: load the published parent labels ----------
def load_reference():
    hits = glob.glob("/kaggle/input/**/lc25000_image_groups.csv", recursive=True)
    if hits:
        print("reference labels found in inputs:", hits[0]); return pd.read_csv(hits[0])
    dst = f"{OUT_DIR}/lc25000_image_groups.csv"
    try:
        urllib.request.urlretrieve(REF_URL, dst)
        print("reference labels downloaded from GitHub"); return pd.read_csv(dst)
    except Exception as e:
        print("download failed:", repr(e)); return None

ref = None if SYNTHETIC else load_reference()

if ref is None:
    if not SYNTHETIC:
        raise RuntimeError(
            "Could not get the LC25000-clean labels. Turn Internet ON in the notebook settings "
            "(or attach lc25000_image_groups.csv as a dataset) and run again. We stop here on "
            "purpose: the KMeans-only split leaks and must not be the headline result.")
    print("!! synthetic data: no reference labels exist; using KMeans groups as a stand-in.")
    df["group_ref"] = df["group_kmeans"]
else:
    need = {"filename", "label", "group_id"}
    assert need <= set(ref.columns), f"reference CSV is missing {need - set(ref.columns)}"
    assert ref.filename.is_unique
    assert ref.groupby("group_id").label.nunique().max() == 1, "a reference parent spans two classes"
    df["filename"] = df.path.map(os.path.basename)
    assert df.filename.is_unique, "two images share a file name"
    j = df.merge(ref[["filename", "label", "group_id"]], on="filename", how="left",
                 validate="one_to_one")
    assert len(j) == len(df)
    missing = int(j.group_id.isna().sum())
    assert missing == 0, f"{missing} images have no reference label (file names differ?)"
    bad = int((j.label != j.cls).sum())
    assert bad == 0, f"{bad} images have a different class in the reference file"
    df["group_ref"] = "ref_" + j["group_id"].astype(int).astype(str).to_numpy()
    print(f"joined {len(df)}/{len(df)} images, every class label agrees")

print("KMeans groups:", df.group_kmeans.nunique(), "| reference groups:", df.group_ref.nunique())
_sz = df.groupby("group_ref").size()
print(f"reference group size: mean {_sz.mean():.1f}, min {_sz.min()}, max {_sz.max()} "
      + ("(about 20 expected on the full dataset)" if not QUICK_TEST else "(QUICK_TEST subset: small groups are normal)"))

reference labels downloaded from GitHub
joined 25000/25000 images, every class label agrees
KMeans groups: 1250 | reference groups: 1246
reference group size: mean 20.1, min 1, max 53 (about 20 expected on the full dataset)


In [6]:
# ---------- step 2b: agreement (ARI audit) ----------
def pair_precision_recall(ours, truth):
    ct = pd.crosstab(ours, truth).to_numpy()
    tp = comb(ct, 2).sum()
    p_den, r_den = comb(ct.sum(1), 2).sum(), comb(ct.sum(0), 2).sum()
    return (tp/p_den if p_den else float("nan")), (tp/r_den if r_den else float("nan"))

_rows = []
for scope in ["global (inflated)"] + CLASSES:
    d = df if scope.startswith("global") else df[df.cls == scope]
    h, c_, v = homogeneity_completeness_v_measure(d.group_ref, d.group_kmeans)
    p, r = pair_precision_recall(d.group_kmeans, d.group_ref)
    _rows.append(dict(scope=scope, n=len(d), kmeans_groups=d.group_kmeans.nunique(),
                      ref_groups=d.group_ref.nunique(),
                      ARI=adjusted_rand_score(d.group_ref, d.group_kmeans),
                      NMI=normalized_mutual_info_score(d.group_ref, d.group_kmeans),
                      homogeneity=h, completeness=c_, v_measure=v,
                      pair_precision=p, pair_recall=r))
agree = pd.DataFrame(_rows)
agree.to_csv(f"{OUT_DIR}/table9_group_agreement.csv", index=False)
print(agree.round(4).to_string(index=False))
if SYNTHETIC: print("(synthetic: reference == KMeans, so every score is trivially 1)")

            scope     n  kmeans_groups  ref_groups    ARI    NMI  homogeneity  completeness  v_measure  pair_precision  pair_recall
global (inflated) 25000           1250        1246 0.5961 0.9202       0.9164        0.9241     0.9202          0.5610       0.6366
         lung_aca  5000            250         243 0.6642 0.9186       0.9163        0.9210     0.9186          0.6372       0.6968
         lung_scc  5000            250         248 0.6885 0.9267       0.9216        0.9320     0.9267          0.6494       0.7356
           lung_n  5000            250         249 0.5608 0.8862       0.8808        0.8917     0.8862          0.5257       0.6053
        colon_aca  5000            250         249 0.6006 0.9013       0.8948        0.9079     0.9013          0.5569       0.6559
          colon_n  5000            250         257 0.4591 0.8506       0.8461        0.8551     0.8506          0.4368       0.4892


In [7]:
# ---------- step 3: splits and leakage ----------
def group_split(frame, col):
    g1 = GroupShuffleSplit(1, test_size=0.30, random_state=SEED)
    a, b = next(g1.split(frame, frame.y, groups=frame[col]))
    trn, tmp_ = frame.iloc[a], frame.iloc[b]
    g2 = GroupShuffleSplit(1, test_size=0.50, random_state=SEED)
    v, t = next(g2.split(tmp_, tmp_.y, groups=tmp_[col]))
    return trn, tmp_.iloc[v], tmp_.iloc[t]

def leak_pct(trn, tst, col="group_ref"):
    """% of tst images whose TRUE parent (reference group) also appears in trn."""
    return 100.0*float(tst[col].isin(set(trn[col])).mean())

rtr, rtmp = train_test_split(df, test_size=0.30, stratify=df.y, random_state=SEED)
rva, rte  = train_test_split(rtmp, test_size=0.50, stratify=rtmp.y, random_state=SEED)
ktr, kva, kte = group_split(df, "group_kmeans")      # audit only
gtr, gva, gte = group_split(df, "group_ref")         # the split we train on

leak = pd.DataFrame([
    {"split": n, "train": len(a), "val": len(v), "test": len(t),
     "test_leak_%_by_kmeans_groups": round(leak_pct(a, t, "group_kmeans"), 2),
     "test_leak_%_by_reference_groups": round(leak_pct(a, t), 2),
     "val_leak_%_by_reference_groups": round(leak_pct(a, v), 2)}
    for n, (a, v, t) in {"random": (rtr, rva, rte), "kmeans_group": (ktr, kva, kte),
                         "reference_group": (gtr, gva, gte)}.items()])
leak.to_csv(f"{OUT_DIR}/table1_leakage_audit.csv", index=False)
print(leak.to_string(index=False))

kleak = (kte.assign(leaked=kte.group_ref.isin(set(ktr.group_ref)))
            .groupby("cls").leaked.agg(leak_rate="mean", leaked="sum", n="count"))
kleak.to_csv(f"{OUT_DIR}/table10_kmeans_leak_per_class.csv")
print("\nwhere the KMeans split leaks (true parent of a test image also in train):")
print(kleak.round(4))

# ---------- choose the training split ----------
if SPLIT_ON == "reference":
    tr, va, te = gtr, gva, gte
elif SPLIT_ON == "kmeans":
    tr, va, te = ktr, kva, kte
    print("\n!! SPLIT_ON='kmeans': this split LEAKS. Use it only to reproduce old numbers.")
else:
    raise ValueError("SPLIT_ON must be 'reference' or 'kmeans'")
df["group"] = df["group_ref"] if SPLIT_ON == "reference" else df["group_kmeans"]
df.drop(columns=[c for c in ["filename"] if c in df]).to_csv(
    f"{OUT_DIR}/image_index_with_groups.csv", index=False)

if SPLIT_ON == "reference":
    for a_, b_, nm in [(tr, te, "train/test"), (tr, va, "train/val"), (va, te, "val/test")]:
        assert not set(a_.group_ref) & set(b_.group_ref), f"{nm} share a parent: split is broken"
    print("\nOK: train, val and test share no original image (reference groups).")
print(f"training split: {SPLIT_ON}  sizes {len(tr)}/{len(va)}/{len(te)}")
print("test class counts:", te.cls.value_counts().reindex(CLASSES).to_dict())

          split  train  val  test  test_leak_%_by_kmeans_groups  test_leak_%_by_reference_groups  val_leak_%_by_reference_groups
         random  17500 3750  3750                        100.00                            99.97                           99.97
   kmeans_group  17270 3779  3951                          0.00                            57.99                           59.62
reference_group  17399 3778  3823                         51.71                             0.00                            0.00

where the KMeans split leaks (true parent of a test image also in train):
           leak_rate  leaked    n
cls                              
colon_aca     0.4946     364  736
colon_n       0.7908     722  913
lung_aca      0.4674     358  766
lung_n        0.7927     585  738
lung_scc      0.3283     262  798

OK: train, val and test share no original image (reference groups).
training split: reference  sizes 17399/3778/3823
test class counts: {'lung_aca': 637, 'lung_scc': 816,

In [8]:
import torchvision.transforms as T
MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]

# augmentation is kept mild: heavy augmentation would partly re-create the duplicate
# problem we just removed
train_tf = T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)), T.RandomHorizontalFlip(),
                      T.RandomVerticalFlip(), T.ColorJitter(0.1,0.1,0.1),
                      T.ToTensor(), T.Normalize(MEAN,STD)])
eval_tf  = T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN,STD)])

class HistoDataset(Dataset):
    def __init__(self, frame, tf):
        self.paths, self.labels, self.tf = frame.path.tolist(), frame.y.tolist(), tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert("RGB")), self.labels[i]

nw = 2 if QUICK_TEST else 4
tr_ld = DataLoader(HistoDataset(tr, train_tf), BATCH_SIZE, shuffle=True,
                   num_workers=nw, pin_memory=True, drop_last=True)
va_ld = DataLoader(HistoDataset(va, eval_tf), BATCH_SIZE, num_workers=nw, pin_memory=True)
te_ld = DataLoader(HistoDataset(te, eval_tf), BATCH_SIZE, num_workers=nw, pin_memory=True)
print("loaders ready")

loaders ready


In [9]:
# ---------- report style (used only by the report_* figures) ----------
REPORT_CLS = {"lung_aca": "Lung ACA", "lung_scc": "Lung SCC", "lung_n": "Lung benign",
              "colon_aca": "Colon ACA", "colon_n": "Colon benign"}
C_BLUE, C_ORANGE, C_AQUA, C_YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
C_INK, C_INK2, C_GRID = "#0b0b0b", "#52514e", "#e4e3df"
REPORT_RC = {"font.size": 10, "axes.edgecolor": "#b9b8b3", "axes.labelcolor": C_INK2,
             "xtick.color": C_INK2, "ytick.color": C_INK2, "axes.titlecolor": C_INK,
             "axes.titlesize": 11, "axes.spines.top": False, "axes.spines.right": False}
def rgrid(ax, axis="y"): ax.grid(axis=axis, color=C_GRID, lw=0.8); ax.set_axisbelow(True)
def rsave(fig, name, svg=False):
    fig.savefig(f"{OUT_DIR}/{name}.png", dpi=200, bbox_inches="tight", facecolor="white")
    if svg: fig.savefig(f"{OUT_DIR}/{name}.svg", bbox_inches="tight", facecolor="white")
    plt.close(fig); print("saved", f"{name}.png" + (" + .svg" if svg else ""))

In [10]:
# ---------- REPORT: dataset samples and augmentation (no random numbers are used here) ----------
import torchvision.transforms.functional as TFv
with plt.rc_context(REPORT_RC):
    fig, axs = plt.subplots(1, 5, figsize=(15, 3.6))
    for a, c in zip(axs, CLASSES):
        p = te[te.cls == c].sort_values("path").path.iloc[0]
        a.imshow(Image.open(p).convert("RGB")); a.set_title(REPORT_CLS[c], fontsize=12); a.axis("off")
    fig.suptitle("LC25000 test-set samples (original 768 × 768 tiles)", fontsize=12, y=1.02)
    rsave(fig, "report_fig_dataset_samples")

    img = Image.open(te[te.cls == "lung_aca"].sort_values("path").path.iloc[0]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    jit = TFv.adjust_saturation(TFv.adjust_contrast(TFv.adjust_brightness(img, 1.1), 0.9), 1.1)
    views = [(img, "Original (resized 224)"), (TFv.hflip(img), "Horizontal flip"),
             (TFv.vflip(img), "Vertical flip"), (jit, "Colour jitter (±0.1)")]
    fig, axs = plt.subplots(1, 4, figsize=(12, 3.3))
    for a, (im, t) in zip(axs, views): a.imshow(im); a.set_title(t, fontsize=11); a.axis("off")
    rsave(fig, "report_fig_augmentation")

saved report_fig_dataset_samples.png
saved report_fig_augmentation.png


## Section 5 — The selective scan

A Mamba layer is the recurrence

$$h_t = \bar{A}_t h_{t-1} + \bar{B}_t u_t, \qquad y_t = C_t h_t + D u_t$$

As a Python loop over 196 tokens this is far too slow to train, so we evaluate it with a
**parallel scan** — the recurrence is associative, so it needs $\log_2$ steps instead of $L$.
That is why this notebook does not need the `mamba-ssm` CUDA package, which is painful to
compile on Kaggle.

**Chunking is what prevents CUDA out-of-memory.** A naive full-length parallel scan builds
`(batch, d_inner, length, d_state)` tensors once per scan step and autograd keeps every one of
them — roughly 7.5 GB across 8 blocks, which kills a 16 GB GPU. We cut the sequence into
`CHUNK_L` blocks: parallel inside a chunk, sequential across chunks, carrying the state with
`exp(cumsum(delta*A))`, which is exact. Combined with gradient checkpointing this drops peak
training memory by about 12x, to under 2 GB.

The cell below also keeps a plain sequential implementation and **asserts the two agree at
every chunk size**. A silent bug here would invalidate the entire thesis, so it is worth the
three seconds.

In [11]:
def _shift_right(x, n, fill):
    if n >= x.shape[2]: return torch.full_like(x, fill)
    return torch.cat([torch.full_like(x[:,:,:n], fill), x[:,:,:-n]], dim=2)

def _inclusive_scan(a, x):
    """T_j = a_j * T_{j-1} + x_j, in log2(L) vectorised steps."""
    l, off = a.shape[2], 1
    while off < l:
        x = x + a*_shift_right(x, off, 0.0)
        a = a*_shift_right(a, off, 1.0)
        off *= 2
    return x

def selective_scan_parallel(u, delta, A, B, C, D, chunk=None):
    """u,delta:(b,d,l)  A:(d,n)  B,C:(b,n,l)  D:(d,) -> y:(b,d,l)
    Parallel inside a chunk, sequential across chunks. Mathematically identical
    to the full scan; the carry exp(cumsum(delta*A)) is exact."""
    chunk = chunk or CHUNK_L
    b, d, l = u.shape
    log_a = delta.unsqueeze(-1)*A[:,None,:]
    x_in  = delta.unsqueeze(-1)*B.transpose(1,2).unsqueeze(1)*u.unsqueeze(-1)
    h_prev, ys = u.new_zeros(b,d,A.shape[-1]), []
    for s0 in range(0, l, chunk):
        s1 = min(s0+chunk, l); la = log_a[:,:,s0:s1]
        h = _inclusive_scan(la.exp(), x_in[:,:,s0:s1]) + la.cumsum(2).exp()*h_prev.unsqueeze(2)
        ys.append(torch.einsum("bdln,bnl->bdl", h, C[:,:,s0:s1]))
        h_prev = h[:,:,-1]
    return torch.cat(ys,2) + u*D[None,:,None]

def selective_scan_sequential(u, delta, A, B, C, D):
    """Slow reference. Only used to verify the fast path."""
    b, d, l = u.shape
    a = torch.exp(delta.unsqueeze(-1)*A[:,None,:])
    x = delta.unsqueeze(-1)*B.transpose(1,2).unsqueeze(1)*u.unsqueeze(-1)
    h = u.new_zeros(b, d, A.shape[-1]); ys = []
    for t in range(l):
        h = a[:,:,t]*h + x[:,:,t]
        ys.append(torch.einsum("bdn,bn->bd", h, C[:,:,t]))
    return torch.stack(ys,-1) + u*D[None,:,None]

# ---- verification --------------------------------------------------------
torch.manual_seed(0)
_u  = torch.randn(2,6,64); _dl = F.softplus(torch.randn(2,6,64))
_A  = -torch.rand(6,4)-0.1; _B = torch.randn(2,4,64)
_C  = torch.randn(2,4,64);  _D = torch.randn(6)
_ref = selective_scan_sequential(_u,_dl,_A,_B,_C,_D)
for _ck in [1,4,7,32,1000]:
    _e = (selective_scan_parallel(_u,_dl,_A,_B,_C,_D,chunk=_ck)-_ref).abs().max().item()
    print(f"  chunk={_ck:<5d} max |chunked - sequential| = {_e:.2e}")
    assert _e < 1e-4, f"Chunked scan wrong at chunk={_ck} - do not trust any result."
print("OK: chunked scan verified against the sequential reference.")

  chunk=1     max |chunked - sequential| = 0.00e+00
  chunk=4     max |chunked - sequential| = 1.91e-06
  chunk=7     max |chunked - sequential| = 1.91e-06
  chunk=32    max |chunked - sequential| = 1.91e-06
  chunk=1000  max |chunked - sequential| = 2.38e-06
OK: chunked scan verified against the sequential reference.


## Section 6 — The model

A deliberately plain bidirectional CNN+Mamba. No attention modules, no handcrafted texture
features, no ensembling — it is the **instrument**, not the contribution. Adding an attention
block would also wreck the argument, since a reviewer would ask whether the explanation came
from the SSM or from the attention.

The CNN stem is kept small on purpose: its receptive field should span a few nuclei, not a
whole gland. Long-range aggregation is the Mamba stack's job.

**`capture=True` returns `C`, the gate and `out_proj` as well as `delta`, `A`, `B`, `u`.** Those
first three are exactly what Run 1's score threw away, and exactly what carries class
information to the output. Note in `forward` that `delta`, `B` and `C` are all computed *from
the input* — which is why our score is an approximation and why Section 10 measures the gap
rather than assuming it away.

A tissue patch has no natural reading order, so each layer runs one forward scan and one
reverse scan. With a single direction the bottom-right of the image would receive top-left
context but not the reverse, which is an artefact of the scan rather than of the biology.

In [12]:
def _silu_lrp(x):
    """Eq. 4: SiLU with the sigmoid detached, restoring relevance conservation."""
    return x * torch.sigmoid(x).detach()


class MambaBlock(nn.Module):
    def __init__(self, d_model, d_state=D_STATE, d_conv=D_CONV, expand=EXPAND):
        super().__init__()
        d_inner = expand*d_model
        self.d_inner, self.d_state, self.d_conv = d_inner, d_state, d_conv
        self.in_proj = nn.Linear(d_model, d_inner*2, bias=False)
        self.conv1d  = nn.Conv1d(d_inner, d_inner, d_conv, groups=d_inner, padding=d_conv-1)
        self.dt_rank = math.ceil(d_model/16)
        self.x_proj  = nn.Linear(d_inner, self.dt_rank+2*d_state, bias=False)
        self.dt_proj = nn.Linear(self.dt_rank, d_inner, bias=True)
        dt = torch.exp(torch.rand(d_inner)*(math.log(0.1)-math.log(1e-3))+math.log(1e-3)).clamp(min=1e-4)
        with torch.no_grad(): self.dt_proj.bias.copy_(dt + torch.log(-torch.expm1(-dt)))
        A = torch.arange(1, d_state+1, dtype=torch.float32).repeat(d_inner,1)
        self.A_log = nn.Parameter(torch.log(A))      # A kept negative -> stable decay
        self.D = nn.Parameter(torch.ones(d_inner))
        self.out_proj = nn.Linear(d_inner, d_model, bias=False)

    def forward(self, x, capture=False):
        b, l, _ = x.shape
        u, z = self.in_proj(x).chunk(2,-1)
        u = F.silu(self.conv1d(u.transpose(1,2))[..., :l])
        x_dbl = self.x_proj(u.transpose(1,2))                    # delta, B, C come FROM u
        delta, B, C = torch.split(x_dbl, [self.dt_rank, self.d_state, self.d_state], -1)
        delta = F.softplus(self.dt_proj(delta)).transpose(1,2)
        B, C = B.transpose(1,2).contiguous(), C.transpose(1,2).contiguous()
        A = -torch.exp(self.A_log.float())
        y = selective_scan_parallel(u, delta, A, B, C, self.D.float())
        gate = F.silu(z.transpose(1,2))
        out = self.out_proj((y*gate).transpose(1,2))
        if capture:
            return out, {"delta":delta.detach(), "A":A.detach(), "B":B.detach(),
                         "C":C.detach(), "u":u.detach(), "gate":gate.detach(),
                         "D":self.D.detach(), "W_out":self.out_proj.weight.detach()}
        return out, None

    def forward_lrp(self, x):
        """MambaLRP forward pass (Jafari et al., NeurIPS 2024).

        Three modifications, exactly as the paper specifies:
          Eq. 4     SiLU        y = x * [sigma(x)]_cst
          Eq. 6-7   SSM         Abar_t, Bbar_t, C_t all treated as constants
          Eq. 9     gate        y = 0.5*(zA*zB) + 0.5*[zA*zB]_cst
        Relevance is then read off as gradient x input. LayerNorm is left standard,
        since the paper's stated rules do not cover it.
        """
        b, l, _ = x.shape
        u, z = self.in_proj(x).chunk(2, -1)
        u = _silu_lrp(self.conv1d(u.transpose(1, 2))[..., :l])
        x_dbl = self.x_proj(u.transpose(1, 2))
        delta, B, C = torch.split(x_dbl, [self.dt_rank, self.d_state, self.d_state], -1)
        # ---- Eq. 6-7: the input-dependent SSM parameters become constants ----
        delta = F.softplus(self.dt_proj(delta)).transpose(1, 2).detach()
        B = B.transpose(1, 2).contiguous().detach()
        C = C.transpose(1, 2).contiguous().detach()
        A = (-torch.exp(self.A_log.float())).detach()
        y = selective_scan_parallel(u, delta, A, B, C, self.D.float())
        # ---- Eq. 9: half-relevance through the multiplicative gate ----
        prod = y * _silu_lrp(z.transpose(1, 2))
        return self.out_proj((0.5*prod + 0.5*prod.detach()).transpose(1, 2))


class BiMambaLayer(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.fwd, self.bwd = MambaBlock(d_model), MambaBlock(d_model)
    def forward(self, x, capture=False):
        h = self.norm(x)
        yf, cf = self.fwd(h, capture); yb, cb = self.bwd(h.flip(1), capture)
        return x + yf + yb.flip(1), ({"fwd":cf,"bwd":cb} if capture else None)
    def forward_lrp(self, x):
        h = self.norm(x)
        return x + self.fwd.forward_lrp(h) + self.bwd.forward_lrp(h.flip(1)).flip(1)


def _ckpt_layer(layer, t): return layer(t, False)[0]

class CNNStem(nn.Module):
    """224 -> 28 via three stride-2 blocks. Local morphology only."""
    def __init__(self, dims=(64,128,128)):
        super().__init__()
        ch, L = (3,)+dims, []
        for i in range(len(dims)):
            L += [nn.Conv2d(ch[i],ch[i+1],3,2,1,bias=False), nn.BatchNorm2d(ch[i+1]), nn.GELU(),
                  nn.Conv2d(ch[i+1],ch[i+1],3,1,1,bias=False), nn.BatchNorm2d(ch[i+1]), nn.GELU()]
        self.net, self.out_dim = nn.Sequential(*L), dims[-1]
    def forward(self, x): return self.net(x)

class CNNMamba(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, d_model=D_MODEL, depth=DEPTH):
        super().__init__()
        self.stem = CNNStem()
        self.patch_embed = nn.Conv2d(self.stem.out_dim, d_model, PATCH, PATCH)
        self.grid = IMG_SIZE//8//PATCH               # 224 -> 28 -> 14 (196 tokens, as ViT-B/16)
        self.num_tokens = self.grid**2
        self.layers = nn.ModuleList([BiMambaLayer(d_model) for _ in range(depth)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)
        self.apply(self._init)
    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
    def tokenize(self, x):
        return self.patch_embed(self.stem(x)).flatten(2).transpose(1,2)
    def from_tokens(self, t, capture=False):
        caps = []
        for layer in self.layers:
            if USE_CHECKPOINT and self.training and not capture and t.requires_grad:
                t = torch.utils.checkpoint.checkpoint(_ckpt_layer, layer, t, use_reentrant=False)
            else:
                t, c = layer(t, capture)
                if capture: caps.append(c)
        return self.head(self.norm(t).mean(1)), caps
    def forward(self, x, capture=False):
        lo, caps = self.from_tokens(self.tokenize(x), capture)
        return (lo, caps) if capture else lo
    def from_tokens_lrp(self, t):
        for layer in self.layers:
            t = layer.forward_lrp(t)
        return self.head(self.norm(t).mean(1))

print("parameters:", sum(p.numel() for p in CNNMamba().parameters())/1e6, "M")

parameters: 2.595781 M


In [13]:
# ---------- REPORT: system diagram (black and white only) ----------
from matplotlib.patches import FancyBboxPatch
_st = torch.get_rng_state()
_nparam = sum(p.numel() for p in CNNMamba().parameters()) / 1e6
torch.set_rng_state(_st)                     # building the model above must not change the random stream
_ngroups = df.group_ref.nunique()
K, Wt = "black", "white"
fig, ax = plt.subplots(figsize=(16, 9.2)); ax.set_xlim(0, 16); ax.set_ylim(0, 9.2); ax.axis("off")

def box(x, y, w, h, title, lines, lw=1.2, ls="-", tsize=10.5, lsize=8.6):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0,rounding_size=0.1", fc=Wt, ec=K, lw=lw, ls=ls, zorder=2))
    top = y + h / 2 + 0.16 * len(lines) + 0.02
    ax.text(x + w / 2, top, title, ha="center", va="center", fontsize=tsize, weight="bold", color=K, zorder=3)
    for i, l in enumerate(lines):
        ax.text(x + w / 2, top - 0.34 - 0.27 * i, l, ha="center", va="center", fontsize=lsize, color=K, zorder=3)
def tag(x, y, text):
    w = 0.108 * len(text) + 0.55
    ax.add_patch(FancyBboxPatch((x, y), w, 0.36, boxstyle="round,pad=0,rounding_size=0.18", fc=K, ec=K, zorder=2))
    ax.text(x + w / 2, y + 0.18, text, ha="center", va="center", fontsize=9, weight="bold", color=Wt, zorder=3)
def arrow(pts, ls="-", lw=1.2):
    for (x0, y0), (x1, y1) in zip(pts[:-2], pts[1:-1]):
        ax.plot([x0, x1], [y0, y1], color=K, lw=lw, ls=ls, solid_capstyle="butt", zorder=1)
    (x0, y0), (x1, y1) = pts[-2], pts[-1]
    ax.annotate("", (x1, y1), (x0, y0), zorder=1, arrowprops=dict(arrowstyle="-|>,head_length=0.5,head_width=0.22",
                color=K, lw=lw, ls=ls, shrinkA=0, shrinkB=0))

tag(0.4, 8.55, "A   DATA")
dx, dw, dh = 0.4, 2.6, 1.15
box(dx, 6.95, dw, dh, "LC25000", ["25,000 H&E images", "5 classes"])
box(dx, 5.45, dw, dh, "Group split", [f"{_ngroups:,} original groups", "70 / 15 / 15, 0 % overlap"])
box(dx, 3.95, dw, dh, "Pre-processing", ["resize 224, flips,", "mild colour jitter"])
arrow([(dx + dw / 2, 6.95), (dx + dw / 2, 6.6)]); arrow([(dx + dw / 2, 5.45), (dx + dw / 2, 5.1)])

tag(3.55, 5.55, f"B   CLASSIFIER  ({_nparam:.2f} M parameters)")
my, mh = 3.9, 1.25
blocks = [(3.55, 1.7, "Input", ["H&E tile", "224 × 224 × 3"], 1.2),
          (5.75, 2.2, "CNN stem", ["3 × (conv s2 + conv)", "→ 28 × 28 × 128"], 1.2),
          (8.45, 2.0, "Patch embed", ["2 × 2 conv", "→ 196 tokens × 192"], 1.2),
          (10.95, 2.3, f"BiMamba × {DEPTH}", ["forward + backward scan", "→ 196 × 192"], 2.4),
          (13.7, 1.9, "Head", ["mean pooling", "linear → 5 classes"], 1.2)]
for x, w, t, l, lw in blocks: box(x, my, w, mh, t, l, lw=lw)
arrow([(dx + dw, my + mh / 2), (3.55, my + mh / 2)])
for (x, w, *_), (x2, *__) in zip(blocks[:-1], blocks[1:]): arrow([(x + w, my + mh / 2), (x2, my + mh / 2)])

zx, zy, zw, zh = 8.45, 6.1, 7.15, 2.75
ax.add_patch(FancyBboxPatch((zx, zy), zw, zh, boxstyle="round,pad=0,rounding_size=0.12", fc=Wt, ec=K, lw=1.0, ls=(0, (4, 3)), zorder=0))
ax.text(zx + 0.2, zy + zh - 0.3, "Inside one BiMamba layer", fontsize=9.5, weight="bold", ha="left", va="center")
ax.plot([10.95, zx + 2.2], [my + mh, zy], color=K, lw=0.8, ls=(0, (2, 3)))
ax.plot([13.25, zx + zw - 0.4], [my + mh, zy], color=K, lw=0.8, ls=(0, (2, 3)))
cy = zy + 1.15
def sbox(x, w, t, y=cy, h=0.62):
    ax.add_patch(FancyBboxPatch((x, y - h / 2), w, h, boxstyle="round,pad=0,rounding_size=0.08", fc=Wt, ec=K, lw=1.0, zorder=2))
    ax.text(x + w / 2, y, t, ha="center", va="center", fontsize=8.6, zorder=3)
ax.text(zx + 0.35, cy, "tokens", ha="left", va="center", fontsize=8.6)
sbox(zx + 1.25, 1.25, "LayerNorm")
sbox(zx + 3.1, 2.05, "Mamba  →  forward", y=cy + 0.5)
sbox(zx + 3.1, 2.05, "Mamba  ←  backward", y=cy - 0.5)
ax.add_patch(plt.Circle((zx + 5.75, cy), 0.2, fc=Wt, ec=K, lw=1.1, zorder=2))
ax.text(zx + 5.75, cy + 0.01, "+", ha="center", va="center", fontsize=12, zorder=3)
arrow([(zx + 1.0, cy), (zx + 1.25, cy)])
arrow([(zx + 2.5, cy), (zx + 2.8, cy), (zx + 2.8, cy + 0.5), (zx + 3.1, cy + 0.5)])
arrow([(zx + 2.8, cy), (zx + 2.8, cy - 0.5), (zx + 3.1, cy - 0.5)])
ax.plot([zx + 5.15, zx + 5.35, zx + 5.35], [cy + 0.5, cy + 0.5, cy], color=K, lw=1.2)
ax.plot([zx + 5.15, zx + 5.35, zx + 5.35], [cy - 0.5, cy - 0.5, cy], color=K, lw=1.2)
arrow([(zx + 5.35, cy), (zx + 5.55, cy)])
arrow([(zx + 0.8, cy + 0.12), (zx + 0.8, zy + zh - 0.62), (zx + 5.75, zy + zh - 0.62), (zx + 5.75, cy + 0.2)])
ax.text(zx + 3.3, zy + zh - 0.56, "residual", ha="center", va="bottom", fontsize=8, style="italic")
arrow([(zx + 5.95, cy), (zx + 6.55, cy)]); ax.text(zx + 6.62, cy, "out", ha="left", va="center", fontsize=8.6)

tag(0.4, 2.95, "C   EXPLANATION & EVALUATION")
ey, eh = 1.2, 1.35
box(0.4, ey, 2.6, eh, "Baselines", ["Grad-CAM, LIME,", "MambaLRP"])
box(5.85, ey, 3.3, eh, "influence_cc  (ours)", ["class vector from head", "+ frozen scan, reverse pass"], lw=2.4)
box(10.0, ey, 2.3, eh, "Class heatmap", ["14 × 14 relevance", "→ 224 × 224"])
box(13.15, ey, 2.45, eh, "Evaluation", ["accuracy, P / R / F1", "stability, sanity, ins / del"])
arrow([(9.15, ey + eh / 2), (10.0, ey + eh / 2)]); arrow([(12.3, ey + eh / 2), (13.15, ey + eh / 2)])
bus = 3.3
ax.plot([12.1, 12.1], [my, bus], color=K, lw=1.1, ls=(0, (4, 3)))
ax.plot([14.65, 14.65], [my, bus], color=K, lw=1.1, ls=(0, (4, 3)))
ax.plot([7.5, 14.65], [bus, bus], color=K, lw=1.1, ls=(0, (4, 3)))
arrow([(7.5, bus), (7.5, ey + eh)], ls=(0, (4, 3)), lw=1.1)
ax.text(10.35, bus + 0.1, "scan values + class weights  (one forward pass, no gradients)", ha="center", va="bottom",
        fontsize=8.3, style="italic", bbox=dict(fc=Wt, ec="none", pad=1))
arrow([(1.7, ey), (1.7, 0.55), (14.375, 0.55), (14.375, ey)])
ax.text(8.0, 0.62, f"same {N_XAI_IMAGES} test images, same metrics", ha="center", va="bottom", fontsize=8.3, style="italic")
rsave(fig, "report_fig_system_diagram", svg=True)

saved report_fig_system_diagram.png + .svg


## Section 7 — Training

AdamW, cosine schedule, cross-entropy with label smoothing, best-validation checkpointing.
Roughly 145 minutes on a T4.

**Weights are saved immediately afterwards.** Every later experiment on this model — adding
MambaLRP, changing a metric, a subgroup analysis — then costs minutes instead of hours.

**Expect accuracy to be lower than earlier runs.** Earlier runs (0.985 to 0.993) used the KMeans
split, which the ARI audit in Section 4 showed still left 58% of test images with a sibling of
the same original image in train. The test set here contains **no** sibling of any training
image, so this number is the honest one. A drop is the expected effect of removing leakage, not a bug.
Report it as your main accuracy.

In [14]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

@torch.no_grad()
def evaluate(m, loader):
    m.eval(); P, Y = [], []
    for x, y in loader:
        o = m(x.to(DEVICE, non_blocking=True))
        if isinstance(o, tuple): o = o[0]
        P.append(o.argmax(1).cpu()); Y.append(y)
    P, Y = torch.cat(P).numpy(), torch.cat(Y).numpy()
    return (accuracy_score(Y,P), f1_score(Y,P,average="macro"),
            confusion_matrix(Y,P,labels=list(range(NUM_CLASSES))))

@torch.no_grad()
def eval_loss_acc(m, loader, crit):
    """ADDED for the report: validation accuracy AND loss in ONE pass over the loader.
    One pass = the same number of DataLoader iterators as the original code, so the
    random-number stream (shuffling, augmentation) stays exactly as in the ARI run."""
    m.eval(); tot = n = cor = 0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        o = m(x)
        if isinstance(o, tuple): o = o[0]
        tot += crit(o, y).item()*y.size(0); n += y.size(0); cor += (o.argmax(1) == y).sum().item()
    return tot/max(n,1), cor/max(n,1)

set_seed(); free_gpu()
model = CNNMamba().to(DEVICE)
opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, EPOCHS))
crit  = nn.CrossEntropyLoss(label_smoothing=0.05)

best, best_state, best_ep, HISTORY = -1.0, None, 0, []
for ep in range(EPOCHS):
    model.train(); tot = n = cor = 0; t0 = time.time()
    for x, y in tr_ld:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        logits = model(x)
        if isinstance(logits, tuple): logits = logits[0]
        loss = crit(logits, y)
        opt.zero_grad(set_to_none=True); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        tot += loss.item()*y.size(0); n += y.size(0)
        cor += (logits.detach().argmax(1) == y).sum().item()          # ADDED: train accuracy
    lr_now = opt.param_groups[0]["lr"]
    sched.step()
    vloss, vacc = eval_loss_acc(model, va_ld, crit)                     # same val_acc as before + val loss
    HISTORY.append(dict(epoch=ep+1, train_loss=tot/max(n,1), train_acc=cor/max(n,1),
                        val_loss=vloss, val_acc=vacc, lr=lr_now, seconds=round(time.time()-t0, 1)))
    pd.DataFrame(HISTORY).to_csv(f"{OUT_DIR}/report_training_history.csv", index=False)
    if vacc > best:
        best, best_ep = vacc, ep+1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    print(f"  epoch {ep+1}/{EPOCHS}  loss {tot/max(n,1):.4f}  train_acc {cor/max(n,1):.4f}  "
          f"val_loss {vloss:.4f}  val_acc {vacc:.4f} ({time.time()-t0:.0f}s)", flush=True)

if best_state: model.load_state_dict(best_state)
acc, mf1, cm = evaluate(model, te_ld)
torch.save(model.state_dict(), f"{OUT_DIR}/weights_{MODEL_NAME}_{SPLIT_NAME}.pt")
np.save(f"{OUT_DIR}/cm_{MODEL_NAME}_{SPLIT_NAME}.npy", cm)
pd.DataFrame([{"model":MODEL_NAME,"split":SPLIT_NAME,
               "test_acc":round(acc,4),"test_macro_f1":round(mf1,4)}]
             ).to_csv(f"{OUT_DIR}/table2_accuracy.csv", index=False)

print(f"\nbest validation epoch: {best_ep}")
print(f"test acc {acc:.4f}   macro-F1 {mf1:.4f}   peak GPU {gpu_peak_gb():.2f} GB")
print("weights saved -> future experiments need no retraining")
model.eval(); free_gpu()

  epoch 1/12  loss 0.5050  train_acc 0.8693  val_loss 0.4112  val_acc 0.9206 (731s)
  epoch 2/12  loss 0.3616  train_acc 0.9378  val_loss 0.3901  val_acc 0.9373 (732s)
  epoch 3/12  loss 0.3085  train_acc 0.9646  val_loss 0.3381  val_acc 0.9685 (731s)
  epoch 4/12  loss 0.2821  train_acc 0.9750  val_loss 0.3621  val_acc 0.9531 (732s)
  epoch 5/12  loss 0.2714  train_acc 0.9802  val_loss 0.5245  val_acc 0.8997 (732s)
  epoch 6/12  loss 0.2575  train_acc 0.9867  val_loss 0.3006  val_acc 0.9674 (732s)
  epoch 7/12  loss 0.2487  train_acc 0.9901  val_loss 0.3314  val_acc 0.9722 (733s)
  epoch 8/12  loss 0.2379  train_acc 0.9947  val_loss 0.3615  val_acc 0.9600 (731s)
  epoch 9/12  loss 0.2327  train_acc 0.9968  val_loss 0.3268  val_acc 0.9685 (732s)
  epoch 10/12  loss 0.2286  train_acc 0.9984  val_loss 0.3321  val_acc 0.9701 (731s)
  epoch 11/12  loss 0.2263  train_acc 0.9990  val_loss 0.3312  val_acc 0.9693 (732s)
  epoch 12/12  loss 0.2256  train_acc 0.9994  val_loss 0.3268  val_acc 0.9

In [15]:
# --- the hard pair, isolated ---
ACA, SCC = CLASS_TO_IDX["lung_aca"], CLASS_TO_IDX["lung_scc"]
pair = cm[[ACA,SCC]][:, [ACA,SCC]]
hard = pd.DataFrame([{
    "aca_vs_scc_acc": round(float(np.trace(pair)/max(pair.sum(),1)),4),
    "aca_called_scc": int(cm[ACA,SCC]), "scc_called_aca": int(cm[SCC,ACA]),
    "total_pair_errors": int(cm[ACA,SCC]+cm[SCC,ACA])}])
hard.to_csv(f"{OUT_DIR}/table3_aca_vs_scc.csv", index=False)

print("confusion matrix (rows=true, cols=pred):")
print("            " + " ".join(f"{c:>10s}" for c in CLASSES))
for i, r in enumerate(cm):
    print(f"{CLASSES[i]:>11s} " + " ".join(f"{v:>10d}" for v in r))
print("\nper-class recall:", {CLASSES[i]: round(float(r[i]/r.sum()),4) for i,r in enumerate(cm)})
print("\n" + hard.to_string(index=False))
print("\nAlmost every error lives in this one pair -- the other three classes are near-perfect.")

confusion matrix (rows=true, cols=pred):
              lung_aca   lung_scc     lung_n  colon_aca    colon_n
   lung_aca        565         70          0          2          0
   lung_scc          2        814          0          0          0
     lung_n          0          0        835          0          0
  colon_aca          0          0          0        749          0
    colon_n          0          0          0          0        786

per-class recall: {'lung_aca': 0.887, 'lung_scc': 0.9975, 'lung_n': 1.0, 'colon_aca': 1.0, 'colon_n': 1.0}

 aca_vs_scc_acc  aca_called_scc  scc_called_aca  total_pair_errors
         0.9504              70               2                 72

Almost every error lives in this one pair -- the other three classes are near-perfect.


In [16]:
# ---------- REPORT: training curves, confusion matrix, classification report, misclassification ----------
# The random-number state is saved here and restored at the end of this cell, so the XAI part
# below runs exactly as in the ARI notebook.
_rng_state = (random.getstate(), np.random.get_state(), torch.get_rng_state(),
              torch.cuda.get_rng_state_all() if DEVICE == "cuda" else None)
from sklearn.metrics import cohen_kappa_score, matthews_corrcoef, precision_recall_fscore_support
H = pd.DataFrame(HISTORY)

# (1) training curves: train vs validation
with plt.rc_context(REPORT_RC):
    fig, axs = plt.subplots(1, 2, figsize=(13, 4.6))
    axs[0].plot(H.epoch, H.train_loss, color=C_BLUE, lw=2.2, marker="o", ms=5, label="train")
    axs[0].plot(H.epoch, H.val_loss, color=C_ORANGE, lw=2.2, marker="s", ms=5, label="validation")
    axs[0].set_title("Loss"); axs[0].set_ylabel("cross-entropy loss (label smoothing 0.05)")
    axs[1].plot(H.epoch, H.train_acc*100, color=C_BLUE, lw=2.2, marker="o", ms=5, label="train")
    axs[1].plot(H.epoch, H.val_acc*100, color=C_ORANGE, lw=2.2, marker="s", ms=5, label="validation")
    axs[1].set_title("Accuracy"); axs[1].set_ylabel("accuracy (%)")
    for a in axs:
        a.axvline(best_ep, color=C_INK2, ls="--", lw=1)
        a.set_xlabel("epoch"); a.set_xticks(H.epoch); rgrid(a); a.legend(frameon=False)
    axs[1].annotate(f"best validation (epoch {best_ep})", (best_ep, H.val_acc.max()*100), xytext=(6, -14),
                    textcoords="offset points", fontsize=8.5, color=C_INK2)
    rsave(fig, "report_fig_training_curves")
print(H.round(4).to_string(index=False))
_last = H.iloc[-1]
print(f"\nlast epoch: val_loss - train_loss = {_last.val_loss - _last.train_loss:+.4f} | "
      f"train_acc - val_acc = {(_last.train_acc - _last.val_acc)*100:+.2f} points")
_rise = int((H.val_loss.diff().iloc[-4:] > 0).sum())
print(f"validation loss went UP in {_rise} of the last 4 epochs "
      "(overfitting = val loss rising for several epochs while train loss keeps falling).")
print("note: train loss/accuracy are measured on augmented images during training.")

# (2) test predictions with probabilities (same model, same order as te)
@torch.no_grad()
def predict_proba(m, loader):
    m.eval(); out = []
    for x, _ in loader:
        o = m(x.to(DEVICE, non_blocking=True))
        if isinstance(o, tuple): o = o[0]
        out.append(F.softmax(o, 1).cpu())
    return torch.cat(out).numpy()
PROBA = predict_proba(model, te_ld)
Y_TE, P_TE = te.y.to_numpy(), PROBA.argmax(1)
assert (confusion_matrix(Y_TE, P_TE, labels=list(range(NUM_CLASSES))) == cm).all(), "prediction mismatch"

# (3) classification report
p, r, f, s = precision_recall_fscore_support(Y_TE, P_TE, labels=list(range(NUM_CLASSES)), zero_division=0)
spec = [(cm.sum() - cm[i].sum() - cm[:, i].sum() + cm[i, i]) / (cm.sum() - cm[i].sum()) for i in range(NUM_CLASSES)]
rep = pd.DataFrame({"class": [REPORT_CLS[c] for c in CLASSES], "precision": p, "recall": r,
                    "specificity": spec, "f1": f, "support": s})
rep.loc[len(rep)] = ["Macro average", p.mean(), r.mean(), np.mean(spec), f.mean(), s.sum()]
rep.round(4).to_csv(f"{OUT_DIR}/report_table_classification.csv", index=False)
overall = pd.DataFrame([{"accuracy": acc, "macro_precision": p.mean(), "macro_recall": r.mean(), "macro_f1": f.mean(),
                         "cohen_kappa": cohen_kappa_score(Y_TE, P_TE), "mcc": matthews_corrcoef(Y_TE, P_TE),
                         "test_images": len(Y_TE), "best_val_epoch": best_ep}])
overall.round(4).to_csv(f"{OUT_DIR}/report_table_overall.csv", index=False)
print("\n=== CLASSIFICATION REPORT (test set) ===")
print(rep.assign(**{k: (rep[k]*100).round(2) for k in ["precision", "recall", "specificity", "f1"]}).to_string(index=False))
print("\n" + overall.round(4).to_string(index=False))

with plt.rc_context(REPORT_RC):
    # (4) confusion matrix
    rowp = cm / cm.sum(1, keepdims=True) * 100
    fig, ax = plt.subplots(figsize=(6.4, 5.4))
    ax.imshow(rowp, cmap="Blues", vmin=0, vmax=100)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, f"{cm[i,j]}\n{rowp[i,j]:.1f}%", ha="center", va="center", fontsize=9,
                    color="white" if rowp[i, j] > 55 else C_INK)
    names = [REPORT_CLS[c] for c in CLASSES]
    ax.set_xticks(range(NUM_CLASSES), names, rotation=25); ax.set_yticks(range(NUM_CLASSES), names)
    ax.set_xlabel("predicted class"); ax.set_ylabel("true class")
    ax.set_title(f"Confusion matrix (n = {cm.sum():,} test images)")
    for sp in ax.spines.values(): sp.set_visible(False)
    rsave(fig, "report_fig_confusion_matrix")

    # (5) precision / recall / F1 per class
    fig, ax = plt.subplots(figsize=(10, 4.4)); xx = np.arange(NUM_CLASSES); w = 0.26
    for k, (vals, lab, col) in enumerate([(p, "Precision", C_BLUE), (r, "Recall", C_ORANGE), (f, "F1-score", C_AQUA)]):
        ax.bar(xx + (k-1)*w, vals*100, w-0.03, color=col, label=lab)
        for xi, v in zip(xx + (k-1)*w, vals*100):
            ax.text(xi, max(v, 80.6) if v > 80 else 80.6, f"{v:.1f}", ha="center", va="bottom", fontsize=7.5,
                    color="white" if v > 83 else C_INK, rotation=90)
    ax.set_xticks(xx, names); ax.set_ylim(80, 101.5); ax.set_ylabel("%")
    ax.set_title(f"Per-class precision, recall and F1-score (test set, n = {cm.sum():,})")
    ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.12)); rgrid(ax)
    rsave(fig, "report_fig_precision_recall_f1")

# (6) misclassification report
wrong = np.flatnonzero(P_TE != Y_TE)
mis = te.iloc[wrong][["path", "cls"]].copy()
mis["predicted"] = [CLASSES[k] for k in P_TE[wrong]]
mis["confidence"] = PROBA[wrong, P_TE[wrong]]
mis["prob_true_class"] = PROBA[wrong, Y_TE[wrong]]
mis = mis.sort_values("confidence", ascending=False)
mis.to_csv(f"{OUT_DIR}/report_misclassified_images.csv", index=False)
pairs = (mis.groupby(["cls", "predicted"]).size().rename("images").reset_index()
            .sort_values("images", ascending=False))
pairs["true"] = pairs.cls.map(REPORT_CLS); pairs["pred"] = pairs.predicted.map(REPORT_CLS)
pairs["share_of_errors_%"] = (pairs.images / len(mis) * 100).round(1)
pairs["share_of_true_class_%"] = [round(n_ / (Y_TE == CLASS_TO_IDX[c_]).sum() * 100, 2)
                                  for n_, c_ in zip(pairs.images, pairs.cls)]
pairs["mean_confidence"] = [mis[(mis.cls == a_) & (mis.predicted == b_)].confidence.mean().round(3)
                            for a_, b_ in zip(pairs.cls, pairs.predicted)]
pairs = pairs[["true", "pred", "images", "share_of_errors_%", "share_of_true_class_%", "mean_confidence"]]
pairs.to_csv(f"{OUT_DIR}/report_table_misclassification.csv", index=False)
print(f"\n=== MISCLASSIFICATION REPORT: {len(mis)} wrong of {len(Y_TE)} test images "
      f"({len(mis)/len(Y_TE)*100:.2f} %) ===")
print(pairs.to_string(index=False))

with plt.rc_context(REPORT_RC):
    pp = pairs.iloc[::-1]
    fig, ax = plt.subplots(figsize=(9, 0.45*len(pp) + 1.2))
    ax.barh([f"{a_} → {b_}" for a_, b_ in zip(pp.true, pp.pred)], pp.images, color=C_BLUE, height=0.6)
    for yy, (v, sh) in enumerate(zip(pp.images, pp["share_of_errors_%"])):
        ax.text(v + max(pp.images)*0.01, yy, f"{v}  ({sh} %)", va="center", fontsize=9, color=C_INK)
    ax.set_xlim(0, max(pp.images)*1.25); ax.set_xlabel("number of wrong test images")
    ax.set_title(f"Where the {len(mis)} errors go (true → predicted)"); rgrid(ax, "x")
    rsave(fig, "report_fig_misclassification")

    show = mis.head(15)
    ncol = 5; nrow = int(np.ceil(len(show)/ncol))
    if len(show):
        fig, axs = plt.subplots(nrow, ncol, figsize=(3*ncol, 3.7*nrow)); axs = np.atleast_1d(axs).ravel()
        for a in axs: a.axis("off")
        for a, (_, rw) in zip(axs, show.iterrows()):
            a.imshow(Image.open(rw.path).convert("RGB").resize((IMG_SIZE, IMG_SIZE)))
            a.set_title(f"true: {REPORT_CLS[rw.cls]}\npred: {REPORT_CLS[rw.predicted]} ({rw.confidence*100:.0f} %)", fontsize=9)
        fig.suptitle("Most confident wrong predictions", fontsize=12)
        plt.tight_layout(); rsave(fig, "report_fig_misclassified_examples")

random.setstate(_rng_state[0]); np.random.set_state(_rng_state[1]); torch.set_rng_state(_rng_state[2])
if _rng_state[3] is not None: torch.cuda.set_rng_state_all(_rng_state[3])

saved report_fig_training_curves.png
 epoch  train_loss  train_acc  val_loss  val_acc     lr  seconds
     1      0.5050     0.8693    0.4112   0.9206 0.0003    731.3
     2      0.3616     0.9378    0.3901   0.9373 0.0003    732.4
     3      0.3085     0.9646    0.3381   0.9685 0.0003    731.3
     4      0.2821     0.9750    0.3621   0.9531 0.0003    732.1
     5      0.2714     0.9802    0.5245   0.8997 0.0002    731.9
     6      0.2575     0.9867    0.3006   0.9674 0.0002    732.5
     7      0.2487     0.9901    0.3314   0.9722 0.0001    732.6
     8      0.2379     0.9947    0.3615   0.9600 0.0001    731.4
     9      0.2327     0.9968    0.3268   0.9685 0.0001    732.3
    10      0.2286     0.9984    0.3321   0.9701 0.0000    731.1
    11      0.2263     0.9990    0.3312   0.9693 0.0000    732.4
    12      0.2256     0.9994    0.3268   0.9698 0.0000    732.7

last epoch: val_loss - train_loss = +0.1011 | train_acc - val_acc = +2.96 points
validation loss went UP in 1 of the 

## Section 8 — The contribution: class-conditional influence

### The problem with the naive score

Run 1's score used `delta`, `A`, `B`, `u` only. It never read `C`, the gate, `out_proj` or the
head, so it **could not depend on the predicted class**. That one fact explains all three of its
failures: near-zero gradient correlation (the true gradient *is* class-specific), a failed
randomisation test (randomising the head cannot change a map that never read it), and — the
trap — the **best** stability scores, because a map driven by the image rather than by the
weights is trivially stable.

### The derivation

With the input-dependent terms frozen for one forward pass, the block is linear in its inputs.
Writing $E(s,t) = \exp(\mathrm{cum}A_t - \mathrm{cum}A_s)$,

$$h_t = \sum_{s \le t} E(s,t)\,\Delta_s B_s u_s, \qquad y_t = C_t h_t + D u_t$$

The gate, `out_proj`, the final `LayerNorm` and mean-pooling are all linear or frozen-linear,
so the **target class weight can be carried straight through them without backpropagation**:

$$a_c = W_{\text{out}}^{\top}\big(\gamma \odot W_{\text{head}}[c]\big) \in \mathbb{R}^{d_{\text{inner}}}$$

giving the class-$c$ relevance of token $s$:

$$R_c[s] = \sum_{d,n} a_c[d]\,S_s[d,n]\,\Delta_s[d]B_s[n]u_s[d],
\qquad S_s = \sum_{t \ge s} g_t \odot C_t \odot E(s,t)$$

### Computing $S_s$ without overflow

The naive form needs $\exp(-\mathrm{cum}A_s)$, which explodes. But $S$ obeys a backward
recursion with the *same* associative structure as the forward scan:

$$S_s = g_s C_s + \bar{A}_{s+1}\,S_{s+1}$$

so we reuse `_inclusive_scan` on the reversed sequence. One extra scan, still $O(L)$, still one
forward pass, **still no gradients** — the property that motivated the method survives intact.

### The variants under test

| name | what it is |
|---|---|
| `influence_rollout` | Run 1's class-agnostic score, the "before" |
| `influence_cc` | class-conditional last layer x earlier-layer rollout prior |
| `influence_cc_last` | the class-conditional term alone, **no prior** |
| `influence_cc_abs` | magnitude instead of ReLU |

`influence_cc_last` matters: Run 2 suggested the rollout prior contributes nothing, which would
mean the method simplifies. Section 10 tests that formally instead of by eye.

The score is **signed** — a token can argue *against* the predicted class — so `relu` keeps
positive evidence (what a clinician wants highlighted) and `abs` keeps magnitude (the fair thing
to correlate against an unsigned gradient reference).

In [17]:
class InfluenceRollout:
    """Run 1's class-agnostic frozen-dynamics score. Kept as the 'before' baseline."""
    def __init__(self, model): self.model = model
    @staticmethod
    def _dir(cap, wbi=True):
        delta, A, B, u = cap["delta"], cap["A"], cap["B"], cap["u"]
        log_a = delta.unsqueeze(-1)*A[:,None,:]
        cum   = log_a.cumsum(2)
        decay = (2.0*(cum[:,:,-1:,:]-cum)).clamp(min=-60.0).exp()
        gain  = (delta.unsqueeze(-1)*B.transpose(1,2).unsqueeze(1))**2
        if wbi: gain = gain*(u.unsqueeze(-1)**2)
        return (decay*gain).sum(dim=(1,3))
    @staticmethod
    def _norm(s):
        s = s - s.amin(1, keepdim=True)
        return s/(s.amax(1, keepdim=True)+1e-8)
    def per_layer(self, x):
        self.model.eval()
        with torch.no_grad(): _, caps = self.model(x, capture=True)
        return [self._norm(self._dir(c["fwd"]) + self._dir(c["bwd"]).flip(1)) for c in caps]
    def __call__(self, x, residual=0.5):
        maps = self.per_layer(x)
        o = torch.ones_like(maps[0])
        for m in maps: o = o*(residual + (1-residual)*m)
        o = self._norm(o).view(-1,1,self.model.grid,self.model.grid)
        return F.interpolate(o, (IMG_SIZE,IMG_SIZE), mode="bilinear", align_corners=False)
    def spatial_spread(self, x):
        """Mean distance of the influence mass from its own centroid, per layer.
        Growing with depth = the Mamba stack really does widen its context."""
        maps = self.per_layer(x); g = self.model.grid
        yy, xx = torch.meshgrid(torch.arange(g).float(), torch.arange(g).float(), indexing="ij")
        coords = torch.stack([yy.flatten(), xx.flatten()], -1)
        out = []
        for m in maps:
            w = (m/(m.sum(1, keepdim=True)+1e-8)).cpu()
            d = (coords.unsqueeze(0) - (w @ coords).unsqueeze(1)).norm(dim=-1)
            out.append(float((w*d).sum(1).mean()))
        return out


class ClassConditionalInfluence:
    """Forward-pass, class-conditional frozen-dynamics relevance (Section 8)."""
    def __init__(self, model, use_prior=True, residual=0.5, mode="relu"):
        self.model, self.use_prior, self.residual, self.mode = model, use_prior, residual, mode
    @staticmethod
    def _reverse_state_sum(cap):
        """S_s = g_s*C_s + A_{s+1} * S_{s+1}, via one reversed associative scan."""
        a = (cap["delta"].unsqueeze(-1)*cap["A"][:,None,:]).exp()
        X = cap["gate"].unsqueeze(-1)*cap["C"].transpose(1,2).unsqueeze(1)
        return _inclusive_scan(_shift_right(a.flip(2),1,1.0), X.flip(2)).flip(2)
    def _dir(self, cap, a_cls):
        S = self._reverse_state_sum(cap)
        delta, B, u, D, gate = cap["delta"], cap["B"], cap["u"], cap["D"], cap["gate"]
        R = (torch.einsum("bdln,bnl->bdl", S, B)*delta*u*a_cls[:,:,None]).sum(1)
        return R + (D[None,:,None]*u*gate*a_cls[:,:,None]).sum(1)      # skip path
    def _class_vector(self, cap, target):
        """a_c = W_out^T (gamma * W_head[c]) -- linear, no autograd."""
        gamma = self.model.norm.weight.detach()
        return (self.model.head.weight.detach()[target]*gamma) @ cap["W_out"]
    @staticmethod
    def _norm(s):
        s = s - s.amin(1, keepdim=True)
        return s/(s.amax(1, keepdim=True)+1e-8)
    def __call__(self, x, target=None):
        self.model.eval()
        with torch.no_grad():
            logits, caps = self.model(x, capture=True)
            if target is None: target = logits.argmax(1)
            last = caps[-1]
            R = (self._dir(last["fwd"], self._class_vector(last["fwd"], target))
                 + self._dir(last["bwd"], self._class_vector(last["bwd"], target)).flip(1))
            R = self._norm(F.relu(R) if self.mode == "relu" else R.abs())
            if self.use_prior and len(caps) > 1:
                prior = torch.ones_like(R)
                for c in caps[:-1]:
                    m = InfluenceRollout._norm(InfluenceRollout._dir(c["fwd"])
                                               + InfluenceRollout._dir(c["bwd"]).flip(1))
                    prior = prior*(self.residual + (1-self.residual)*m)
                R = self._norm(R*prior)
        g = self.model.grid
        return F.interpolate(R.view(-1,1,g,g), (IMG_SIZE,IMG_SIZE),
                             mode="bilinear", align_corners=False)
print("explainers defined")

explainers defined


## Section 9 — Baselines and metrics

Grad-CAM on the token grid, at the same resolution as ours. LIME implemented from scratch with
SLIC superpixels and ridge regression, so a missing package cannot break a four-hour run.

### MambaLRP (Jafari et al., NeurIPS 2024) — the baseline that matters

MambaLRP shows that naive Layer-wise Relevance Propagation breaks on Mamba, because three
components violate relevance conservation. It fixes each one and then reads relevance off as
gradient x input on the corrected forward pass. We implement its three stated rules exactly:

| paper | component | rule |
|---|---|---|
| Eq. 4 | SiLU | `y = x * sigmoid(x).detach()` — the sigmoid becomes a constant |
| Eq. 6–7 | selective SSM | `A_bar`, `B_bar`, `C` all detached; gradient flows only through the state path |
| Eq. 9 | multiplicative gate | `y = 0.5*(zA*zB) + 0.5*(zA*zB).detach()` — stops relevance doubling |

**Two honest caveats.** First, the official code targets language Mamba and Vim; this is our own
port to a bidirectional vision block with a CNN stem, so it is a reimplementation, not the
authors' code — the next cell validates it and the paper should say this plainly. Second, the
paper's stated rules do not cover LayerNorm, so ours is left standard.

**And the observation worth putting in your discussion:** MambaLRP detaches `A_bar`, `B_bar` and
`C` for the same reason our score freezes them — propagating through input-dependent SSM
parameters is unsound. Two independent lines of work reach the same conclusion. That turns our
"frozen-dynamics approximation" from an admitted shortcut into a principled design choice.

**Stability.** A clinically irrelevant change should not move the explanation. We rotate, add
noise and jitter colour at three severities — and crucially **rotate the map back** before
comparing, otherwise a perfectly correct explanation that rotates with the tissue would be
scored as unstable.

Two further controls matter here. The **target class is fixed** at the clean prediction for
both maps: if the perturbed image could pick its own argmax, a prediction flip would make us
compare explanations of two different classes and call the difference "instability". And we
report the **prediction-flip rate** separately, because a perturbation that routinely changes
the diagnosis is not "clinically irrelevant" and the premise of the test would fail.

**Faithfulness.** Stability alone is worthless: a constant heat map is perfectly stable and
explains nothing. *Deletion* hides the top-scoring regions first (lower is better); *insertion*
reveals them first (higher is better). The baseline is a blur rather than black, because black
patches are out of distribution for a stained slide and the drop would measure the artefact.

**Sanity.** Randomising the weights must destroy the explanation. If a map survives that, it
is an edge detector, not an explanation. Many published attribution methods fail this.

Two scopes are reported, and the distinction is not cosmetic:

- **`sanity_iou_full`** — CNN stem **and** patch embedding **and** every Mamba layer **and**
  the head. This is the honest whole-network test and **the number to quote**.
- **`sanity_iou_mamba`** — Mamba layers + head only, leaving the CNN stem trained. An earlier
  version of this study used that narrower scope; it is kept so the two are comparable, but it
  is a weaker test, because a trained stem can carry structure through on its own.

In both cases the **target class is held fixed** at the clean model's prediction. A randomised
network's own argmax is arbitrary, and letting it choose would confound the comparison.

**If the method scores high on `sanity_iou_full`, report it — that is a real weakness, not
something to bury.**

> `deterministic == True` is **not** evidence of stability. Grad-CAM is deterministic in eval
> mode and LIME is deterministic with a fixed seed. Only the measured numbers below support
> any claim.

> `map_std` guards against a trap: an all-zero map scores a perfect IoU of 1.0 and passes every
> stability test while explaining nothing. If a row's `map_std` is near zero, ignore its IoU.

In [18]:
def grad_cam(model, x, target=None):
    model.eval()
    t = model.tokenize(x); t.requires_grad_(True)
    logits, _ = model.from_tokens(t, capture=False)
    if target is None: target = logits.argmax(1)
    grad = torch.autograd.grad(logits.gather(1, target[:,None]).sum(), t)[0]
    cam = F.relu((grad*t).sum(-1))
    cam = cam - cam.amin(1, keepdim=True); cam = cam/(cam.amax(1, keepdim=True)+1e-8)
    g = model.grid
    return F.interpolate(cam.view(-1,1,g,g), (IMG_SIZE,IMG_SIZE),
                         mode="bilinear", align_corners=False).detach()

def lime_map(model, x, target=None, n_samples=None, n_segments=50):
    from skimage.segmentation import slic
    from sklearn.linear_model import Ridge
    n_samples = n_samples or LIME_SAMPLES
    model.eval(); outs = []
    with torch.no_grad():
        bl = model(x)
        if isinstance(bl, tuple): bl = bl[0]
    tgt = bl.argmax(1) if target is None else target
    for i in range(x.shape[0]):
        img = x[i:i+1]; arr = img[0].permute(1,2,0).cpu().numpy()
        seg = slic(arr, n_segments=n_segments, compactness=10.0, channel_axis=-1, start_label=0)
        seg_t = torch.from_numpy(seg).to(x.device); k = int(seg.max())+1
        masks = np.random.binomial(1,0.5,(n_samples,k)).astype(np.float32); masks[0] = 1.0
        fill = img.mean(dim=(2,3), keepdim=True); probs = []
        with torch.no_grad():
            for b0 in range(0, n_samples, 16):
                batch = []
                for m in masks[b0:b0+16]:
                    keep = torch.from_numpy(m).to(x.device)[seg_t][None,None]
                    batch.append(img*keep + fill*(1-keep))
                lo = model(torch.cat(batch,0))
                if isinstance(lo, tuple): lo = lo[0]
                probs.append(F.softmax(lo,1)[:, tgt[i]].cpu().numpy())
        w = np.clip(Ridge(alpha=1.0).fit(masks, np.concatenate(probs)).coef_, 0, None)
        m = torch.from_numpy(w).float().to(x.device)[seg_t]
        m = m - m.min(); outs.append((m/(m.max()+1e-8))[None,None])
    return torch.cat(outs, 0)

def mamba_lrp(model, x, target=None):
    """MambaLRP relevance: gradient x input on the conservation-corrected forward pass."""
    model.eval()
    t = model.tokenize(x).detach().requires_grad_(True)
    logits = model.from_tokens_lrp(t)
    if target is None: target = logits.argmax(1)
    grad = torch.autograd.grad(logits.gather(1, target[:, None]).sum(), t)[0]
    R = F.relu((grad * t).sum(-1))                 # positive relevance, as Grad-CAM does
    R = R - R.amin(1, keepdim=True); R = R/(R.amax(1, keepdim=True)+1e-8)
    g = model.grid
    return F.interpolate(R.view(-1,1,g,g), (IMG_SIZE,IMG_SIZE),
                         mode="bilinear", align_corners=False).detach()


def plain_grad_input(model, x, target=None):
    """Gradient x input on the UNMODIFIED forward pass. Only used to prove that the
    MambaLRP detachments actually change something -- see the validation cell."""
    model.eval()
    t = model.tokenize(x).detach().requires_grad_(True)
    logits, _ = model.from_tokens(t, capture=False)
    if target is None: target = logits.argmax(1)
    grad = torch.autograd.grad(logits.gather(1, target[:, None]).sum(), t)[0]
    R = F.relu((grad * t).sum(-1)).detach()
    R = R - R.amin(1, keepdim=True); return R/(R.amax(1, keepdim=True)+1e-8)


# Every method now takes an EXPLICIT target class. This matters: under perturbation
# the prediction can flip, and comparing an explanation for class A against one for
# class B would measure the flip, not the method. The clean image's predicted class
# is therefore fixed and reused everywhere.
# influence_rollout ignores the target by construction -- it is class-agnostic, which
# is precisely the defect the class-conditional variants were built to fix.
METHODS = {
    "influence_rollout": lambda m,x,t: InfluenceRollout(m)(x),
    "influence_cc":      lambda m,x,t: ClassConditionalInfluence(m, use_prior=True)(x, t),
    "influence_cc_last": lambda m,x,t: ClassConditionalInfluence(m, use_prior=False)(x, t),
    "influence_cc_abs":  lambda m,x,t: ClassConditionalInfluence(m, mode="abs")(x, t),
    "grad_cam":          lambda m,x,t: grad_cam(m, x, t),
    "lime":              lambda m,x,t: lime_map(m, x, t),
    "mamba_lrp":         lambda m,x,t: mamba_lrp(m, x, t),
}
print("methods:", list(METHODS))

methods: ['influence_rollout', 'influence_cc', 'influence_cc_last', 'influence_cc_abs', 'grad_cam', 'lime', 'mamba_lrp']


### Validating the MambaLRP implementation

A reimplemented baseline is worthless — worse than none — if it is silently wrong, because the
comparison then flatters us for the wrong reason. Two checks run before it is used.

**0. The port must be behaviourally correct.** Two hard assertions run first. The LRP forward
pass must produce **identical logits** to the normal one — `.detach()` changes gradients, never
values, so any difference means we would be explaining a different model. And after a backward
through the LRP path, `x_proj.weight.grad` and `dt_proj.weight.grad` must still be `None`: those
layers feed only `delta`, `B` and `C`, which Eq. 6–7 makes constants, so any gradient reaching
them proves the rule is not in effect.

**1. The detachments must actually do something.** MambaLRP differs from plain gradient x input
*only* through the three detachments. If the two produce identical maps, the modifications are
not taking effect and there is a bug. We report the rank correlation between them: it should be
high (same family of method) but clearly below 1.

**2. Relevance should be approximately conserved.** That is the axiom the paper is built on. We
report `sum(relevance) / logit`. It will not be exactly 1 here — our network has biases
(BatchNorm, conv and `dt_proj` bias, LayerNorm, the classifier bias) which absorb relevance, and
the paper's rules do not cover LayerNorm. A ratio in a sane range indicates the propagation is
behaving; a ratio near zero or wildly large indicates it is not.

If either check fails, drop the row and say in the paper that the port could not be validated.
An honest absence beats an unfair comparison.

In [19]:
from scipy.stats import spearmanr          # (also imported later, harmless)

def _signed_relevance(model, x, target, lrp):
    """Signed token relevance = (grad x input), on the LRP forward pass or the normal one."""
    model.eval()
    t = model.tokenize(x).detach().requires_grad_(True)
    logits = model.from_tokens_lrp(t) if lrp else model.from_tokens(t, capture=False)[0]
    g = torch.autograd.grad(logits.gather(1, target[:, None]).sum(), t)[0]
    return (g * t).sum(-1).detach(), logits.gather(1, target[:, None]).squeeze(1).detach()

_xb = globals()["X"][:4] if "X" in globals() else next(iter(te_ld))[0][:4].to(DEVICE)
with torch.no_grad():
    _lo = model(_xb)
    if isinstance(_lo, tuple): _lo = _lo[0]
    _tgt = _lo.argmax(1)

# --- check 0a: the LRP forward must give the SAME logits ---------------------
# .detach() changes gradients, never values. If the logits differ, we would be
# explaining a different model than the one we evaluated.
with torch.no_grad():
    _t0 = model.tokenize(_xb)
    _ln, _ = model.from_tokens(_t0.clone(), capture=False)
    _ll    = model.from_tokens_lrp(_t0.clone())
_ferr = (_ln - _ll).abs().max().item()
print(f"check 0a max |normal logits - lrp logits| = {_ferr:.3e}")
assert _ferr < 1e-3, ("The MambaLRP forward pass changes the model's output. "
                      "It would explain a different model -- do not use it.")
print("         the LRP pass alters gradients only, not predictions.")

# --- check 0b: gradient must NOT reach the detached SSM projections -----------
# x_proj and dt_proj feed only delta, B and C. Under Eq. 6-7 those are constants,
# so after a backward through the LRP path their .grad must stay None.
_probe = copy.deepcopy(model)
_tp = _probe.tokenize(_xb).detach().requires_grad_(True)
_probe.from_tokens_lrp(_tp).gather(1, _tgt[:, None]).sum().backward()
_blk = _probe.layers[0].fwd
_leak = (_blk.x_proj.weight.grad is not None) or (_blk.dt_proj.weight.grad is not None)
print(f"check 0b gradient leaked into delta/B/C projections? {_leak}")
assert not _leak, "Eq. 6-7 is not in effect: gradient still flows through Abar/Bbar/C."
print("         Abar, Bbar and C are true constants in the backward pass.")
del _probe, _tp

_R_lrp,   _logit = _signed_relevance(model, _xb, _tgt, lrp=True)
_R_plain, _      = _signed_relevance(model, _xb, _tgt, lrp=False)

# --- check 1: the three detachments must change the attribution ---
_same = torch.allclose(_R_lrp, _R_plain, atol=1e-6)
_reldiff = ((_R_lrp - _R_plain).abs().sum() / (_R_plain.abs().sum() + 1e-12)).item()
print(f"check 1  relative difference from plain grad x input = {_reldiff:.4f}")
assert not _same, ("MambaLRP is identical to plain gradient x input -- "
                   "the detachments are not active. Do not use this baseline.")
print("         the Eq.4 / Eq.6-7 / Eq.9 detachments are active.")

_rhos = [spearmanr(_R_lrp[i].cpu().numpy(), _R_plain[i].cpu().numpy()).correlation
         for i in range(_xb.shape[0])]
_rho = float(np.nanmean(_rhos))
print(f"         Spearman(MambaLRP, plain) = {_rho:.4f}  "
      f"(same family of method, so high is expected; 1.0 would mean no effect)")

# --- check 2: approximate relevance conservation ---
_ratio = (_R_lrp.sum(1) / _logit).cpu().numpy()
print(f"\ncheck 2  sum(relevance)/logit per image = {np.round(_ratio, 3)}")
print(f"         mean {_ratio.mean():.3f}. Not exactly 1: biases (BatchNorm, conv, dt_proj,")
print( "         LayerNorm, the classifier bias) absorb relevance, and the paper's rules")
print( "         do not cover LayerNorm. Quote this number in the paper.")
if QUICK_TEST:
    print("         (QUICK_TEST: the model is barely trained, so its logits are near zero and")
    print("          this ratio is meaningless. Read it from the real run.)")
print("\nMambaLRP port validated -- safe to use as a baseline.")
del _R_lrp, _R_plain, _logit
free_gpu()

check 0a max |normal logits - lrp logits| = 1.192e-07
         the LRP pass alters gradients only, not predictions.
check 0b gradient leaked into delta/B/C projections? False
         Abar, Bbar and C are true constants in the backward pass.
check 1  relative difference from plain grad x input = 0.7706
         the Eq.4 / Eq.6-7 / Eq.9 detachments are active.
         Spearman(MambaLRP, plain) = 0.6796  (same family of method, so high is expected; 1.0 would mean no effect)

check 2  sum(relevance)/logit per image = [0.05  0.036 0.004 0.032]
         mean 0.031. Not exactly 1: biases (BatchNorm, conv, dt_proj,
         LayerNorm, the classifier bias) absorb relevance, and the paper's rules
         do not cover LayerNorm. Quote this number in the paper.

MambaLRP port validated -- safe to use as a baseline.


In [20]:
import torchvision.transforms.functional as TF
from scipy.stats import spearmanr

PERTURB = {"low":dict(angle=5.0,noise=0.02,bright=0.00),
           "mid":dict(angle=10.0,noise=0.05,bright=0.10),
           "high":dict(angle=20.0,noise=0.10,bright=0.20)}

def perturb(x, cfg):
    o = TF.rotate(x, cfg["angle"])
    if cfg["bright"] > 0: o = o*(1.0+cfg["bright"])
    return o + torch.randn_like(o)*cfg["noise"]

def iou_topk(a, b, frac=TOP_FRAC):
    n = a.shape[0]; a, b = a.reshape(n,-1), b.reshape(n,-1)
    k = max(1, int(frac*a.shape[1]))
    ma = torch.zeros_like(a).scatter_(1, a.topk(k,1).indices, 1.0)
    mb = torch.zeros_like(b).scatter_(1, b.topk(k,1).indices, 1.0)
    return ((ma*mb).sum(1)/(((ma+mb)>0).float().sum(1)+1e-8)).cpu().numpy()

def stability(model, fn, x, level, target):
    """IoU between the clean map and the rotation-ALIGNED perturbed map.

    Both maps are computed for the SAME target class (the clean prediction). If the
    perturbed image were allowed to pick its own argmax, a prediction flip would make
    us compare explanations of two different classes -- measuring the flip, not the
    method's stability."""
    cfg = PERTURB[level]
    m0 = fn(model, x, target)
    m1 = TF.rotate(fn(model, perturb(x, cfg), target), -cfg["angle"])
    c = int(IMG_SIZE*0.15)                       # drop borders damaged by rotation
    return iou_topk(m0[...,c:-c,c:-c], m1[...,c:-c,c:-c])

@torch.no_grad()
def prediction_flip_rate(model, x, target, level):
    """How often perturbation changes the predicted class. Reported as a diagnostic:
    a high rate means the perturbation is not 'clinically irrelevant' after all."""
    lo = model(perturb(x, PERTURB[level]))
    if isinstance(lo, tuple): lo = lo[0]
    return (lo.argmax(1) != target).float().cpu().numpy()

@torch.no_grad()
def _curve(model, x, attr, target, steps, mode):
    n = x.shape[0]
    blur = F.conv2d(x, torch.ones(3,1,11,11,device=x.device)/121.0, padding=5, groups=3)
    order = attr.reshape(n,-1).argsort(1, descending=True); total = order.shape[1]
    ps = []
    for i in range(steps+1):
        k = int(total*i/steps)
        if mode == "deletion":
            m = torch.ones(n,total,device=x.device)
            if k: m.scatter_(1, order[:,:k], 0.0)
        else:
            m = torch.zeros(n,total,device=x.device)
            if k: m.scatter_(1, order[:,:k], 1.0)
        m = m.view(n,1,IMG_SIZE,IMG_SIZE)
        lo = model(x*m + blur*(1-m))
        if isinstance(lo, tuple): lo = lo[0]
        ps.append(F.softmax(lo,1).gather(1, target[:,None]).squeeze(1))
    return torch.stack(ps,1)

def sanity_check(model, fn, x, target, scope="full"):
    """Model-parameter randomisation (Adebayo et al.). LOW is good.

    scope="full"  -> CNN stem + patch embedding + every Mamba layer + head.
                     This is the honest 'whole network' test and the one to quote.
    scope="mamba" -> Mamba layers + head only; the CNN stem is left trained.
                     Reported alongside because the earlier version of this study
                     used that narrower scope, and the two are not comparable.

    The target class is held fixed at the clean model's prediction; a randomised
    model's own argmax is meaningless and would confound the comparison.
    """
    m0 = fn(model, x, target)
    rnd = copy.deepcopy(model)
    if scope == "full":
        mods = [rnd.stem, rnd.patch_embed] + list(rnd.layers) + [rnd.head]
    else:
        mods = list(rnd.layers) + [rnd.head]
    for mod in mods:
        for p in mod.parameters(): nn.init.normal_(p, std=0.05)
    m1 = fn(rnd, x, target); del rnd
    return iou_topk(m0, m1)

def true_gradient_map(model, x, target=None):
    model.eval()
    t = model.tokenize(x).detach().requires_grad_(True)
    logits, _ = model.from_tokens(t, capture=False)
    if target is None: target = logits.argmax(1)
    g = torch.autograd.grad(logits.gather(1, target[:,None]).sum(), t)[0]
    s = g.abs().sum(-1); s = s - s.amin(1, keepdim=True)
    return s/(s.amax(1, keepdim=True)+1e-8)

METRICS = ["deletion_auc","insertion_auc","iou_low","iou_mid","iou_high",
           "sanity_iou_full","sanity_iou_mamba","spearman_vs_gradient"]
HIGHER_IS_BETTER = {"deletion_auc":False, "insertion_auc":True, "iou_low":True,
                    "iou_mid":True, "iou_high":True, "sanity_iou_full":False,
                    "sanity_iou_mamba":False, "spearman_vs_gradient":True}
print("metrics:", METRICS)

metrics: ['deletion_auc', 'insertion_auc', 'iou_low', 'iou_mid', 'iou_high', 'sanity_iou_full', 'sanity_iou_mamba', 'spearman_vs_gradient']


## Section 10 — Per-image evaluation

### The sampling fix

An earlier version of this study drew its XAI images by taking the first `N` batches of the
test loader. The test set is ordered by class and `shuffle=False`, so **all 200 evaluation
images turned out to be `lung_aca`** — the comparison was real, but it described one class, not
five. The assertion in the cell below makes that failure impossible: the sample is drawn
**`PER_CLASS` images from each of the five classes**, and the run stops if the result is not
balanced.

Because the sample is now stratified, `table7_xai_per_class.csv` also reports every metric
broken down by class — which is a stronger result than a single pooled number.

Nothing is averaged away here: every metric is kept **per image**, which is what makes the
paired tests in Section 11 possible, and what makes any later re-analysis free.

`set_seed(SEED)` is called before each method so the perturbations and LIME's mask sampling are
identical across methods. Without it, part of the difference between methods would be sampling
noise rather than a property of the explanations — and the paired test would be invalid.

In [21]:
free_gpu()

# ---------------------------------------------------------------------------
# STRATIFIED sample: PER_CLASS images from EACH of the five classes.
# The test set is ordered by class, so taking the first N images would have
# produced an all-lung_aca sample. The assertion below makes that impossible.
# ---------------------------------------------------------------------------
_rng = np.random.RandomState(SEED)
_y = te.y.to_numpy()
_sel = []
for _ci in range(NUM_CLASSES):
    _pos = np.flatnonzero(_y == _ci)
    _k = min(PER_CLASS, len(_pos))
    _sel.extend(_rng.choice(_pos, size=_k, replace=False).tolist())
_sel = np.array(sorted(_sel))
xai_df = te.iloc[_sel].reset_index(drop=True)

xai_ld = DataLoader(HistoDataset(xai_df, eval_tf), BATCH_SIZE, shuffle=False,
                    num_workers=nw, pin_memory=True)
_X, _Y = [], []
for x, y in xai_ld:
    _X.append(x); _Y.append(y)
X = torch.cat(_X).to(DEVICE); Y = torch.cat(_Y).to(DEVICE)
CLS_OF_IMG = xai_df.cls.tolist()

_counts = pd.Series(Y.cpu().numpy()).value_counts().sort_index()
print("XAI sample composition:")
for _ci, _c in enumerate(CLASSES):
    print(f"   {_c:10s} {int(_counts.get(_ci, 0)):4d}")
assert len(_counts) == NUM_CLASSES, "XAI sample is missing a class entirely."
assert int(_counts.max() - _counts.min()) <= 1, "XAI sample is not class-balanced."
print(f"   total      {X.shape[0]:4d}   -> balanced across all {NUM_CLASSES} classes\n")

with torch.no_grad():
    lo = model(X)
    if isinstance(lo, tuple): lo = lo[0]
    PRED = lo.argmax(1)            # the CLEAN prediction; reused as the fixed target
print(f"evaluating {X.shape[0]} test images x {len(METHODS)} methods")

CHUNK = 4
per_image, CURVES, rows = {}, {}, []
for name, fn in METHODS.items():
    set_seed(SEED)                       # identical perturbations / LIME masks per method
    t0 = time.time(); acc = defaultdict(list)
    for i in range(0, X.shape[0], CHUNK):
        xb, pb = X[i:i+CHUNK], PRED[i:i+CHUNK]
        attr = fn(model, xb, pb)
        dc = _curve(model,xb,attr,pb,DELETION_STEPS,"deletion").cpu().numpy()
        ic = _curve(model,xb,attr,pb,DELETION_STEPS,"insertion").cpu().numpy()
        acc["_del_curve"].append(dc); acc["_ins_curve"].append(ic)
        acc["deletion_auc"].append(dc.mean(1)); acc["insertion_auc"].append(ic.mean(1))
        for lv in PERTURB:
            acc[f"iou_{lv}"].append(stability(model, fn, xb, lv, pb))
        acc["sanity_iou_full"].append(sanity_check(model, fn, xb, pb, "full"))
        acc["sanity_iou_mamba"].append(sanity_check(model, fn, xb, pb, "mamba"))
        acc["map_std"].append(attr.std(dim=(1,2,3)).cpu().numpy())
        g = true_gradient_map(model, xb, pb)
        gg = F.interpolate(g.view(-1,1,model.grid,model.grid), (IMG_SIZE,IMG_SIZE),
                           mode="bilinear", align_corners=False)
        rho = []
        for j in range(xb.shape[0]):
            r = spearmanr(attr[j].flatten().cpu().numpy(), gg[j].flatten().cpu().numpy()).correlation
            rho.append(0.0 if np.isnan(r) else r)
        acc["spearman_vs_gradient"].append(np.array(rho))
    CURVES[name] = (np.concatenate(acc["_del_curve"]).mean(0),
                    np.concatenate(acc["_ins_curve"]).mean(0))
    per_image[name] = {k: np.concatenate(v) for k,v in acc.items() if not k.startswith("_")}
    row = {"method": name}
    row.update({k: round(float(v.mean()),4) for k,v in per_image[name].items()})
    row["minutes"] = round((time.time()-t0)/60, 1)
    if row["map_std"] < 1e-4:
        print(f"  !! WARNING: {name} produced a near-CONSTANT map; its IoU is meaningless.")
    rows.append(row)
    print(f"{name:20s}", {k: row[k] for k in row if k != "method"}, flush=True)

xai = pd.DataFrame(rows)
xai.to_csv(f"{OUT_DIR}/table4_xai_comparison.csv", index=False)

recs = [{"image":i, "cls":CLS_OF_IMG[i], "method":nm, "metric":k, "value":float(v)}
        for nm,d in per_image.items() for k,arr in d.items() for i,v in enumerate(arr)]
long_df = pd.DataFrame(recs)
long_df.to_csv(f"{OUT_DIR}/per_image_metrics.csv", index=False)

# --- per-class breakdown: now possible because the sample is stratified ---
per_cls = (long_df.groupby(["metric","method","cls"])["value"].mean()
           .reset_index().pivot_table(index=["metric","method"], columns="cls", values="value"))
per_cls.round(4).to_csv(f"{OUT_DIR}/table7_xai_per_class.csv")
print("\n=== TABLE 7: XAI metrics broken down by class (head) ===")
print(per_cls.round(3).head(14).to_string())

# --- diagnostic: how often does perturbation flip the prediction? ---
flip = {}
for lv in PERTURB:
    fr = []
    for i in range(0, X.shape[0], CHUNK):
        fr.append(prediction_flip_rate(model, X[i:i+CHUNK], PRED[i:i+CHUNK], lv))
    flip[lv] = float(np.concatenate(fr).mean())
pd.DataFrame([flip]).to_csv(f"{OUT_DIR}/table8_prediction_flip_rate.csv", index=False)
print("\nprediction-flip rate under perturbation:",
      {k: f"{v:.3f}" for k,v in flip.items()})
print("   (high values would mean the perturbation is not clinically irrelevant;")
print("    the stability test holds the target fixed regardless, see Section 9)")

pd.set_option("display.width", 220, "display.max_columns", 50)
print("\n================ TABLE 4: explanation quality ================")
print("iou_*      higher = more stable     | deletion   LOWER  = more faithful")
print("insertion  HIGHER = more faithful   | sanity_*   LOWER  = passes randomisation")
print("map_std near 0 = DEGENERATE map, ignore that row's IoU")
print(xai.to_string(index=False))
print(f"\nsaved {len(recs)} per-image values -> any later analysis costs seconds")

XAI sample composition:
   lung_aca     40
   lung_scc     40
   lung_n       40
   colon_aca    40
   colon_n      40
   total       200   -> balanced across all 5 classes

evaluating 200 test images x 7 methods
influence_rollout    {'deletion_auc': 0.8531, 'insertion_auc': 0.8565, 'iou_low': 0.6732, 'iou_mid': 0.5579, 'iou_high': 0.4279, 'sanity_iou_full': 0.636, 'sanity_iou_mamba': 0.6379, 'map_std': 0.07, 'spearman_vs_gradient': 0.1015, 'minutes': 3.1}
influence_cc         {'deletion_auc': 0.8379, 'insertion_auc': 0.8627, 'iou_low': 0.5671, 'iou_mid': 0.419, 'iou_high': 0.3049, 'sanity_iou_full': 0.0715, 'sanity_iou_mamba': 0.1033, 'map_std': 0.1424, 'spearman_vs_gradient': -0.1103, 'minutes': 3.2}
influence_cc_last    {'deletion_auc': 0.8376, 'insertion_auc': 0.8634, 'iou_low': 0.5673, 'iou_mid': 0.419, 'iou_high': 0.3047, 'sanity_iou_full': 0.0694, 'sanity_iou_mamba': 0.1044, 'map_std': 0.1545, 'spearman_vs_gradient': -0.1226, 'minutes': 3.1}
influence_cc_abs     {'deletion_auc':

## Section 11 — Paired significance tests

Table 4 gives means. Means cannot tell you whether a gap of 0.004 between `influence_cc` and
LIME is real. These tests can.

- **Wilcoxon signed-rank** — paired, because every method saw the identical images, and
  non-parametric, because AUCs are bounded in [0,1] and skewed so a *t*-test's normality
  assumption is unsafe.
- **Holm–Bonferroni**, applied **within each metric family, across the six pairwise
  comparisons** against the reference method. With eight metrics that is 48 tests in total,
  corrected as eight families of six — *not* as one family of 48. State it that way in the
  paper; "48 Holm-corrected tests" would invite a follow-up question.
- **Rank-biserial correlation** as effect size: the share of images favouring one method.
  0 means no difference; ±1 means one method wins on every single image.
- **Bootstrap 95% CI** of the mean paired difference, so the *size* of an effect is visible in
  the metric's own units rather than hidden behind a *p*-value.

A non-significant result is reported as such. **"No detectable difference from LIME at 1/18 of
the cost" is a strong claim, and an honest one.**

In [22]:
from scipy.stats import wilcoxon, rankdata

def rank_biserial(d):
    d = d[d != 0]
    if len(d) == 0: return 0.0
    r = rankdata(np.abs(d)); wp, wm = r[d>0].sum(), r[d<0].sum()
    return float((wp-wm)/(wp+wm)) if (wp+wm) > 0 else 0.0

def boot_ci(d, n=N_BOOTSTRAP, alpha=0.05):
    rng = np.random.default_rng(SEED)
    means = d[rng.integers(0, len(d), size=(n, len(d)))].mean(1)
    return float(np.percentile(means, 100*alpha/2)), float(np.percentile(means, 100*(1-alpha/2)))

def holm(pvals):
    p = np.asarray(pvals, float); n = len(p)
    adj = np.empty(n); run = 0.0
    for rank, i in enumerate(np.argsort(p)):
        run = max(run, (n-rank)*p[i]); adj[i] = min(run, 1.0)
    return adj

stat_rows = []
for metric in METRICS:
    others = [m for m in METHODS if m != REFERENCE]
    raw = []
    for other in others:
        a, b = per_image[REFERENCE][metric], per_image[other][metric]
        d = a - b
        try: _, p = wilcoxon(a, b, zero_method="zsplit")
        except ValueError: p = 1.0
        lo_ci, hi_ci = boot_ci(d)
        raw.append(dict(metric=metric, reference=REFERENCE, other=other,
                        ref_mean=round(float(a.mean()),4), other_mean=round(float(b.mean()),4),
                        mean_diff=round(float(d.mean()),4),
                        ci95_low=round(lo_ci,4), ci95_high=round(hi_ci,4),
                        effect_size=round(rank_biserial(d),3), p_raw=float(p),
                        favours="reference" if (d.mean()>0)==HIGHER_IS_BETTER[metric] else other))
    for r, padj in zip(raw, holm([r["p_raw"] for r in raw])):
        r["p_holm"] = float(f"{padj:.3g}"); r["p_raw"] = float(f"{r['p_raw']:.3g}")
        r["significant"] = bool(padj < ALPHA)
        stat_rows.append(r)

stats_df = pd.DataFrame(stat_rows)
stats_df.to_csv(f"{OUT_DIR}/table5_significance.csv", index=False)

for metric in METRICS:
    sub = stats_df[stats_df.metric == metric]
    print(f"\n=== {metric}  ({'higher' if HIGHER_IS_BETTER[metric] else 'LOWER'} is better) ===")
    print(sub[["other","ref_mean","other_mean","mean_diff","ci95_low","ci95_high",
               "effect_size","p_holm","significant","favours"]].to_string(index=False))


=== deletion_auc  (LOWER is better) ===
            other  ref_mean  other_mean  mean_diff  ci95_low  ci95_high  effect_size  p_holm  significant           favours
influence_rollout    0.8379      0.8531    -0.0152   -0.0261    -0.0047       -0.039  1.0000        False         reference
influence_cc_last    0.8379      0.8376     0.0004   -0.0002     0.0009        0.112  0.6760        False influence_cc_last
 influence_cc_abs    0.8379      0.8395    -0.0016   -0.0040     0.0003       -0.028  1.0000        False         reference
         grad_cam    0.8379      0.8533    -0.0154   -0.0241    -0.0072       -0.221  0.0402         True         reference
             lime    0.8379      0.8269     0.0110   -0.0008     0.0236        0.194  0.0859        False              lime
        mamba_lrp    0.8379      0.8389    -0.0010   -0.0120     0.0099       -0.040  1.0000        False         reference

=== insertion_auc  (higher is better) ===
            other  ref_mean  other_mean  mean_di

In [23]:
print("="*80)
print(f"VERDICT  (reference = {REFERENCE}, alpha = {ALPHA}, Holm-corrected within each metric)")
print("="*80)
for metric in METRICS:
    print(f"\n{metric}  ({'higher' if HIGHER_IS_BETTER[metric] else 'lower'} is better)")
    for _, r in stats_df[stats_df.metric == metric].iterrows():
        rng = f"95% CI [{r.ci95_low:+.4f}, {r.ci95_high:+.4f}]"
        if not r.significant:
            print(f"   vs {r.other:20s} NO detectable difference "
                  f"(diff {r.mean_diff:+.4f}, {rng}, p = {r.p_holm:.3g})")
        else:
            who = REFERENCE if r.favours == "reference" else r.other
            print(f"   vs {r.other:20s} {who} is BETTER "
                  f"(diff {r.mean_diff:+.4f}, {rng}, r = {r.effect_size:+.3f}, p = {r.p_holm:.3g})")
if QUICK_TEST:
    print("\n(QUICK_TEST: with so few images the smallest possible Holm-corrected p is large, "
          "so ignore these p-values. They become meaningful at PER_CLASS = 40.)")

VERDICT  (reference = influence_cc, alpha = 0.05, Holm-corrected within each metric)

deletion_auc  (lower is better)
   vs influence_rollout    NO detectable difference (diff -0.0152, 95% CI [-0.0261, -0.0047], p = 1)
   vs influence_cc_last    NO detectable difference (diff +0.0004, 95% CI [-0.0002, +0.0009], p = 0.676)
   vs influence_cc_abs     NO detectable difference (diff -0.0016, 95% CI [-0.0040, +0.0003], p = 1)
   vs grad_cam             influence_cc is BETTER (diff -0.0154, 95% CI [-0.0241, -0.0072], r = -0.221, p = 0.0402)
   vs lime                 NO detectable difference (diff +0.0110, 95% CI [-0.0008, +0.0236], p = 0.0859)
   vs mamba_lrp            NO detectable difference (diff -0.0010, 95% CI [-0.0120, +0.0099], p = 1)

insertion_auc  (higher is better)
   vs influence_rollout    NO detectable difference (diff +0.0061, 95% CI [-0.0030, +0.0153], p = 1)
   vs influence_cc_last    influence_cc_last is BETTER (diff -0.0007, 95% CI [-0.0013, -0.0002], r = -0.281, p = 0.0

## Section 12 — Figures

1. **Faithfulness curves**, averaged over all evaluation images — not over the first four,
   which is a mistake that makes every curve look flat.
2. **Paired differences**, as box plots. A box straddling zero is a non-significant result you
   can *see* rather than infer from a *p*-value.
3. **Qualitative maps**, before and after class-conditioning.
4. **Spatial spread per layer** — if influence widens with depth, you have *measured* the
   division of labour between the CNN stem and the Mamba stack rather than asserting
   "CNN is local, Mamba is global", which is the sloppy sentence every paper in this area uses.

In [24]:
def denorm(t):
    m = torch.tensor(MEAN).view(3,1,1); s = torch.tensor(STD).view(3,1,1)
    return (t.cpu()*s+m).clamp(0,1).permute(1,2,0).numpy()

# --- fig 1: faithfulness curves (cached in Section 10, not recomputed) ---
fig, axs = plt.subplots(1, 2, figsize=(12,4.5))
for name, (d, ii) in CURVES.items():
    t = np.linspace(0,1,len(d))
    axs[0].plot(t, d, marker="o", ms=3, label=name)
    axs[1].plot(t, ii, marker="o", ms=3, label=name)
axs[0].set_title(f"Deletion, lower is better (n={X.shape[0]})"); axs[0].set_xlabel("fraction removed")
axs[1].set_title(f"Insertion, higher is better (n={X.shape[0]})"); axs[1].set_xlabel("fraction revealed")
for a in axs: a.set_ylabel("p(predicted class)"); a.legend(fontsize=8); a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig1_faithfulness.png", dpi=150); plt.close()
np.save(f"{OUT_DIR}/faithfulness_curves.npy", CURVES, allow_pickle=True)

# --- fig 2: paired differences ---
plot_metrics = [m for m in METRICS if m != "spearman_vs_gradient"]
fig, axes = plt.subplots(2, 3, figsize=(15,8))
for ax, metric in zip(axes.ravel(), plot_metrics):
    others = [m for m in METHODS if m != REFERENCE]
    data = [per_image[REFERENCE][metric] - per_image[o][metric] for o in others]
    bp = ax.boxplot(data, labels=[o.replace("influence_","infl_") for o in others],
                    showmeans=True, patch_artist=True)
    for patch in bp["boxes"]: patch.set_facecolor("#9ecae1"); patch.set_alpha(0.7)
    ax.axhline(0, color="black", lw=1.2, ls="--")
    ax.set_title(f"{metric}\n({'higher' if HIGHER_IS_BETTER[metric] else 'lower'} is better)", fontsize=10)
    ax.set_ylabel(f"{REFERENCE} - other"); ax.tick_params(axis="x", rotation=25, labelsize=8)
    ax.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig2_paired_differences.png", dpi=150); plt.close()

# --- fig 3: qualitative maps ---
n_show = min(4, X.shape[0])
fig, axes = plt.subplots(n_show, len(METHODS)+1, figsize=(3*(len(METHODS)+1), 3*n_show))
axes = np.atleast_2d(axes)
for r in range(n_show):
    axes[r,0].imshow(denorm(X[r])); axes[r,0].set_ylabel(CLASSES[int(Y[r])], fontsize=8)
    if r == 0: axes[r,0].set_title("input", fontsize=9)
    for c, (nm, fn) in enumerate(METHODS.items(), start=1):
        a = fn(model, X[r:r+1], PRED[r:r+1])[0,0].detach().cpu().numpy()
        axes[r,c].imshow(denorm(X[r])); axes[r,c].imshow(a, cmap="jet", alpha=0.45)
        if r == 0: axes[r,c].set_title(nm, fontsize=8)
for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/fig3_maps.png", dpi=150); plt.close()

# --- spatial spread ---
spread = InfluenceRollout(model).spatial_spread(X[:min(16, X.shape[0])])
pd.DataFrame([{f"spread_layer{i+1}": round(s,3) for i,s in enumerate(spread)}]
             ).to_csv(f"{OUT_DIR}/table6_spatial_spread.csv", index=False)
print("spatial spread per layer:", [round(s,2) for s in spread],
      " (should INCREASE with depth)")

print("\nsaved:")
for f in sorted(os.listdir(OUT_DIR)): print("  ", f)

spatial spread per layer: [6.52, 2.82, 5.84, 5.53]  (should INCREASE with depth)

saved:
   cm_cnn_mamba_refgroup.npy
   faithfulness_curves.npy
   fig1_faithfulness.png
   fig2_paired_differences.png
   fig3_maps.png
   image_index_with_groups.csv
   lc25000_image_groups.csv
   per_image_metrics.csv
   report_fig_augmentation.png
   report_fig_confusion_matrix.png
   report_fig_dataset_samples.png
   report_fig_misclassification.png
   report_fig_misclassified_examples.png
   report_fig_precision_recall_f1.png
   report_fig_system_diagram.png
   report_fig_system_diagram.svg
   report_fig_training_curves.png
   report_misclassified_images.csv
   report_table_classification.csv
   report_table_misclassification.csv
   report_table_overall.csv
   report_training_history.csv
   table10_kmeans_leak_per_class.csv
   table1_leakage_audit.csv
   table2_accuracy.csv
   table3_aca_vs_scc.csv
   table4_xai_comparison.csv
   table5_significance.csv
   table6_spatial_spread.csv
   table7_xai_per_

In [25]:
# ---------- REPORT: XAI figures with the four methods only ----------
REPORT_METHODS = [("influence_cc", "influence_cc (ours)", C_BLUE, "o"), ("grad_cam", "Grad-CAM", C_ORANGE, "s"),
                  ("lime", "LIME", C_AQUA, "^"), ("mamba_lrp", "MambaLRP", C_YELLOW, "D")]

# (1) heatmaps: one correctly classified image per class
rows_ = []
for ci in range(NUM_CLASSES):
    idx = [i for i in range(X.shape[0]) if int(Y[i]) == ci and int(PRED[i]) == ci]
    if not idx: idx = [i for i in range(X.shape[0]) if int(Y[i]) == ci]
    if idx: rows_.append(idx[0])
fig, axes = plt.subplots(len(rows_), 5, figsize=(15, 3.05*len(rows_)))
axes = np.atleast_2d(axes)
for r_, i in enumerate(rows_):
    axes[r_, 0].imshow(denorm(X[i])); axes[r_, 0].set_ylabel(REPORT_CLS[CLASSES[int(Y[i])]], fontsize=12)
    for c_, (m_, lab, *_ ) in enumerate(REPORT_METHODS, start=1):
        a_ = METHODS[m_](model, X[i:i+1], PRED[i:i+1])[0, 0].detach().cpu().numpy()
        axes[r_, c_].imshow(denorm(X[i])); axes[r_, c_].imshow(a_, cmap="jet", alpha=0.45)
for c_, t_ in enumerate(["Input"] + [m[1] for m in REPORT_METHODS]): axes[0, c_].set_title(t_, fontsize=12, weight="bold")
for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); rsave(fig, "report_fig_heatmaps")

with plt.rc_context(REPORT_RC):
    # (2) stability vs perturbation
    fig, ax = plt.subplots(figsize=(7.5, 4.3)); lv = ["iou_low", "iou_mid", "iou_high"]
    vals = {m_: [per_image[m_][k].mean() for k in lv] for m_, *_ in REPORT_METHODS}
    order = sorted(vals, key=lambda m_: -vals[m_][-1]); ends = {}
    for k_, m_ in enumerate(order):            # spread the end labels so they never overlap
        y_ = vals[m_][-1]
        if k_ and ends[order[k_-1]] - y_ < 0.012: y_ = ends[order[k_-1]] - 0.012
        ends[m_] = y_
    for m_, lab, col, mk in REPORT_METHODS:
        ax.plot([0, 1, 2], vals[m_], color=col, lw=2, marker=mk, ms=8, label=lab)
        ax.annotate(lab, (2, vals[m_][-1]), xytext=(2.1, ends[m_]), textcoords="data", va="center", fontsize=9, color=C_INK)
    ax.set_xticks([0, 1, 2], ["low", "mid", "high"]); ax.set_xlim(-0.2, 2.9)
    ax.set_xlabel("perturbation strength"); ax.set_ylabel("stability IoU (higher = more stable)")
    ax.set_title(f"Explanation stability under perturbation ({X.shape[0]} images)")
    ax.legend(frameon=False, fontsize=8, loc="upper right"); rgrid(ax)
    rsave(fig, "report_fig_stability")

    # (3) deletion / insertion curves
    fig, axs = plt.subplots(1, 2, figsize=(11, 4))
    for m_, lab, col, mk in REPORT_METHODS:
        d_, i_ = CURVES[m_]; t_ = np.linspace(0, 1, len(d_))
        axs[0].plot(t_, d_, color=col, lw=2, marker=mk, ms=4, markevery=3, label=lab)
        axs[1].plot(t_, i_, color=col, lw=2, marker=mk, ms=4, markevery=3, label=lab)
    axs[0].set_title("Deletion curve (lower area = better)"); axs[0].set_xlabel("fraction of image removed")
    axs[1].set_title("Insertion curve (higher area = better)"); axs[1].set_xlabel("fraction of image revealed")
    for a in axs: a.set_ylabel("p(predicted class)"); a.legend(frameon=False, fontsize=8); rgrid(a)
    rsave(fig, "report_fig_faithfulness")

# (4) summary table: means + result of influence_cc vs each baseline (from table5)
t5 = pd.read_csv(f"{OUT_DIR}/table5_significance.csv")
REP_METRICS = [("iou_low", "Stability, low", "higher"), ("iou_mid", "Stability, mid", "higher"),
               ("iou_high", "Stability, high", "higher"), ("sanity_iou_full", "Sanity, full model", "lower"),
               ("sanity_iou_mamba", "Sanity, Mamba part", "lower"), ("insertion_auc", "Insertion AUC", "higher"),
               ("deletion_auc", "Deletion AUC", "lower")]
tab, W_, T_, L_ = [], 0, 0, 0
for k, lab, better in REP_METRICS:
    row = {"metric": lab, "better": better}
    for m_, mlab, *_ in REPORT_METHODS: row[mlab] = round(float(per_image[m_][k].mean()), 3)
    res = []
    for m_ in ["grad_cam", "lime", "mamba_lrp"]:
        x_ = t5[(t5.metric == k) & (t5.other == m_)].iloc[0]
        v_ = ("win" if x_.favours == "reference" else "loss") if bool(x_.significant) else "tie"
        W_ += v_ == "win"; T_ += v_ == "tie"; L_ += v_ == "loss"; res.append(v_)
    row["ours vs Grad-CAM / LIME / MambaLRP"] = " / ".join(res)
    tab.append(row)
tab = pd.DataFrame(tab); tab.to_csv(f"{OUT_DIR}/report_table_xai.csv", index=False)
print(tab.to_string(index=False))
print(f"\nrecord of influence_cc: {W_} wins, {T_} ties, {L_} losses "
      f"(paired Wilcoxon, Holm-corrected, alpha = {ALPHA})")

saved report_fig_heatmaps.png
saved report_fig_stability.png
saved report_fig_faithfulness.png
            metric better  influence_cc (ours)  Grad-CAM  LIME  MambaLRP ours vs Grad-CAM / LIME / MambaLRP
    Stability, low higher                0.567     0.452 0.252     0.508                    win / win / win
    Stability, mid higher                0.419     0.313 0.235     0.360                    win / win / win
   Stability, high higher                0.305     0.216 0.210     0.257                    win / win / win
Sanity, full model  lower                0.072     0.118 0.135     0.114                    win / win / win
Sanity, Mamba part  lower                0.103     0.103 0.274     0.101                    tie / win / tie
     Insertion AUC higher                0.863     0.872 0.886     0.873                   tie / loss / tie
      Deletion AUC  lower                0.838     0.853 0.827     0.839                    win / tie / tie

record of influence_cc: 14 wins, 6 ties,

## Section 13 — How to read this

### Files

| file | where it goes in the thesis |
|---|---|
| `table1_leakage_audit.csv` | leakage of random, KMeans-group and reference-group splits |
| `table9_group_agreement.csv` | ARI / NMI / pair recall of KMeans groups vs LC25000-clean |
| `table10_kmeans_leak_per_class.csv` | where the KMeans split leaked, by class |
| `table2_accuracy.csv`, `cm_*.npy` | classifier performance |
| `table3_aca_vs_scc.csv` | the hard-pair analysis |
| `table4_xai_comparison.csv` | **the main results table** (7 methods, incl. MambaLRP) |
| `table5_significance.csv` | **the claim that survives review** |
| `per_image_metrics.csv` | every later re-analysis, for free |
| `fig1`–`fig3` | figures |
| `weights_*.pt` | no retraining ever again |

### Reporting checklist

- Quote **p, effect size and CI together**. A *p*-value alone says nothing about whether a
  difference matters.
- Say that correction was applied, and which (Holm, within each metric family).
- Report non-significant results plainly. "We could not detect a difference" is a different
  claim from "comparable" — do not write the second when you mean the first.
- Quote the cost alongside: `influence_cc` takes ~3 min against LIME's ~50 min on the same
  hardware and images.
- **Report `spearman_vs_gradient` even though it does not favour us.** The score is a
  frozen-dynamics approximation and does not track the exact derivative; its value is argued
  from deletion, insertion and randomisation. Saying so first is much better than having a
  reviewer find it.

- **State that MambaLRP is a reimplementation**, name the three rules you implemented, note
  that LayerNorm was left standard, and quote the two validation numbers. Reviewers accept a
  reimplemented baseline that is documented; they do not accept one that is silently assumed
  correct.

### Still outstanding

1. **Class imbalance** (100:1, 1000:1) — real clinics are nothing like a balanced dataset.
2. **A reader study** — do the maps point at tissue a pathologist would call relevant? Even
   20–30 images scored by one clinician is worth more to a medical venue than another metric.

In [26]:
print("all saved files:")
for f_ in sorted(os.listdir(OUT_DIR)): print("  ", f_)

all saved files:
   cm_cnn_mamba_refgroup.npy
   faithfulness_curves.npy
   fig1_faithfulness.png
   fig2_paired_differences.png
   fig3_maps.png
   image_index_with_groups.csv
   lc25000_image_groups.csv
   per_image_metrics.csv
   report_fig_augmentation.png
   report_fig_confusion_matrix.png
   report_fig_dataset_samples.png
   report_fig_faithfulness.png
   report_fig_heatmaps.png
   report_fig_misclassification.png
   report_fig_misclassified_examples.png
   report_fig_precision_recall_f1.png
   report_fig_stability.png
   report_fig_system_diagram.png
   report_fig_system_diagram.svg
   report_fig_training_curves.png
   report_misclassified_images.csv
   report_table_classification.csv
   report_table_misclassification.csv
   report_table_overall.csv
   report_table_xai.csv
   report_training_history.csv
   table10_kmeans_leak_per_class.csv
   table1_leakage_audit.csv
   table2_accuracy.csv
   table3_aca_vs_scc.csv
   table4_xai_comparison.csv
   table5_significance.csv
   table6